<a href="https://colab.research.google.com/github/caguazaco/pronostico_generacion_SIN/blob/main/Pronostico_generacion_SIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pronóstico de la generación eléctrica del Sistema Interconectado Nacional (SIN) a corto plazo (siete días) mediante codificación visual de series temporales y aprendizaje profundo

---

### Descripción del Proyecto

Este documento presenta el desarrollo del pipeline de procesamiento para el pronóstico de corto plazo (7 días) de la generación de energía eléctrica del SIN de Colombia, empleando las series históricas de generación hidráulica y térmica, así como los precios de carbón y gas natural (nacional e importado). Estas series temporales son transformadas en representaciones visuales por medio de técnicas de codificación como Gramian Angular Fields (GAF), Markov Transition Fields (MTF), Recurrence Plots (RP), Escalogramas (CWT) y Espectrogramas (STFT).

### Variable objetivo
- Generación total de energía eléctrica del SIN

### Variables auxiliares
- Serie histórica de generación de energía eléctrica del SIN a partir de plantas hidroeléctricas (GWh)
- Serie histórica de generación de energía eléctrica del SIN a partir de plantas termoeléctricas alimentadas por gas natural (GWh)
- Serie histórica de generación de energía eléctrica del SIN a partir de plantas termoeléctricas alimentadas por carbón (GWh)
- Serie histórica del precio promedio ponderado de energía eléctrica en bolsa nacional (COP/kWh)
- Serie histórica del precio de oferta de gas natural nacional (COP/kWh)
- Serie histórica del precio de oferta de carbón (COP/kWh)

### Fuentes de datos
- XM S.A. E.S.P. (Operador del SIN)
- Período: 01 de enero de 2006 a 31 de diciembre de 2025
- Granularidad: Diaria

## 1. Importación de librerías y configuraciones generales

In [ ]:
# Instalación de librerías especializadas
!pip install keras-tuner -q # Optimización de hiperparámetros
!pip install pyts -q # GAF, MTF, RP
!pip install pywavelets -q # CWT / Scalogram
!pip install stumpy -q # Perfil de matriz

print('Instalación de librerías finalizada')

In [ ]:
# Importación de librerías generales
import keras
import keras_tuner as kt
import gc
import matplotlib.pyplot as plt # Visualización de datos
import matplotlib.dates as mdates
import numpy as np # Manipulación de datos
import os
import pandas as pd # Manipulación de datos
import seaborn as sns # Visualización de datos
import shutil
import signal
import tensorflow as tf
import warnings

# Deep Learning
from keras import layers, models, ops # Operaciones de backend para capas de modelos ViT
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam, RMSprop, SGD # Optimizadores
from keras_tuner import HyperParameters

# Machine Learning
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

# Google Drive
from google.colab import drive, files

print(f'Librerías importadas correctamente')

In [ ]:
# Configuraciones generales

# Advertencias
warnings.filterwarnings('ignore')

# Semillas para garantizar reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# Estilo de gráficas
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Opciones de visualización de DataFrames
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)

# Dimensiones
FIG_W, FIG_H = 7.16, 2.6
FS_LABEL, FS_TICK, FS_LEGEND = 10, 8, 8

plt.rcParams.update({
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'stix',
    'font.size': FS_LABEL,
    'axes.labelsize': FS_LABEL,
    'xtick.labelsize': FS_TICK,
    'ytick.labelsize': FS_TICK,
    'legend.fontsize': FS_LEGEND,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
    'axes.unicode_minus': False,
    'axes.spines.top': False,
    'axes.spines.right': False})

print('Configuraciones aplicadas correctamente')

## 2. Lectura y consolidación de datos

In [ ]:
# Ruta para leer datos
base_path = '' # Indicar ruta base
data_path = '' # Indicar ruta de archivo de datos

# Unidad de medida de variables de generación eléctrica
generation_unit = 'GWh'

# Unidad de medida de variables de precios
price_unit = 'COP/kWh'

In [ ]:
# Monta el contenedor de drive
# drive.mount('/content/drive')

In [ ]:
# Lectura de datos
df_data = pd.read_excel(f'{data_path}', parse_dates = ['fecha'], index_col = 'fecha')

# Vista previa de datos
df_data.head()

## 3. Segmentación de datos y preparación de datos



### 3.1 Definición de variables

In [ ]:
# Variable objetivo a pronosticar
target_col = 'y'

# Variables predictoras
# Ingresan como canales a cada "rama" de los modelos
# Considera variable objetivo (autoregresión)
feature_cols = ['generacion_hidraulica', 'generacion_gas', 'generacion_carbon', 'precio_gas', 'precio_carbon', 'ppp_bolsa', 'y']
# feature_cols = ['y']

# Variables de generación eléctrica
generation_cols = ['generacion_hidraulica', 'generacion_gas', 'generacion_carbon', 'y']
# generation_cols = ['y']

In [ ]:
# Proporciones para segmentación de datos
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 1 - train_ratio - val_ratio
split_names = ['train', 'val', 'test']

# Para estabilizar el rango/varianza ante crecimiento estructural de algunas variables
log_diff_eps = 1e-6 # Evita log(0)

# Parámetros para la construcción de ventanas deslizantes
window_size = 60 # Tamaño de la ventana temporal (días)
horizon = 7 # Horizonte de pronóstico (días)
stride = 7 # Deslizamiento de las ventanas temporales (días)

# Parámetros generales para entrenamiento de modelos
MAX_TRIALS = 10
EPOCHS = 100
BATCH_SIZE = 32
EARLY_STOP_PATIENCE = 15
REDUCE_LR_PATIENCE = 10

BAND_CONFIDENCE_A = 0.90
BAND_CONFIDENCE_B = 0.80
# BAND_WINDOWS = [210, 28, 7] # Número de días de pronóstico a presentar
BAND_WINDOWS = [28] # Número de días de pronóstico a presentar
N_CALIBRATION = 50

# Paleta de colores bandas conformales
orange_base = '#E66101'

band_configs = {
    0.90: {'alpha': 0.15, 'label': 'IP 90 %'},
    0.80: {'alpha': 0.30, 'label': 'IP 80 %'}}

### 3.2 Definición de funciones

In [ ]:
# Calcula el número de días para cada conjunto de entrenamiento, validación y prueba
# Maximiza el número de días a utilizar
def calculate_window_based_split(n_total_days):
  ratios = {'train': train_ratio, 'val': val_ratio, 'test': test_ratio}
  cursor = 0 # Índice del día donde inicia el conjunto actual
  remaining_days = n_total_days
  carry = 0 # Días sobrantes traspasados del conjunto anterior
  result = {}

  for split_name in split_names:
    is_last = (split_name == 'test')

    if is_last:
      target_days = remaining_days # El último conjunto toma todos los días que queden
    else:
      target_days = int(round(n_total_days * ratios[split_name])) + carry
      target_days = min(target_days, remaining_days)

    # Número de ventanas completas posibles
    if target_days < window_size + horizon:
      n_windows = 0
      used_days = 0
    else:
      n_windows = (target_days - window_size - horizon) // stride + 1
      used_days = (n_windows - 1) * stride + (window_size + horizon)

    leftover = target_days - used_days

    result[split_name] = {
          'start_day': cursor,
          'end_day': cursor + used_days - 1,
          'n_days': used_days,
          'n_windows': n_windows,
          'leftover_days': leftover}

    cursor += used_days
    remaining_days -= used_days
    carry = leftover if not is_last else 0 # El sobrante del conjunto de prueba no se traspasa

  return result

# Segmenta el conjunto de datos en entrenamiento, validación y prueba
def split_data(df, split_info):
  df = df.sort_index()
  splits = {}

  for split_name, info in split_info.items():
    i0, i1 = info['start_day'], info['end_day']
    splits[split_name] = df.iloc[i0: i1 + 1].copy()

  # Retorna diccionario con llaves 'train', 'val' y 'test', cada una un DataFrame
  return splits

# Construye los pares (ventana, horizonte) para cada conjunto de datos
# Opera exclusivamente en cada conjunto de datos, evitando fuga de datos
def generate_windows_and_horizons(df_split, feature_cols = feature_cols, target_col = target_col):
  n_days = len(df_split)
  n_windows_expected = ((n_days - window_size - horizon) // stride) + 1 if n_days >= window_size + horizon else 0

  X_list = []
  y_list = []
  window_start_list = []
  window_end_list = []
  horizon_start_list = []
  horizon_end_list = []

  features_arr = df_split[feature_cols].to_numpy()
  target_arr = df_split[target_col].to_numpy()
  dates = df_split.index.to_numpy() # El índice del DataFrame es la fecha

  for k in range(n_windows_expected):
    t0 = k * stride # Inicio de la ventana de entrada
    t_end_window = t0 + window_size # Inicio del horizonte
    t_end_horizon = t_end_window + horizon # Fin del horizonte

    X_list.append(features_arr[t0: t_end_window])
    y_list.append(target_arr[t_end_window: t_end_horizon])

    window_start_list.append(dates[t0])
    window_end_list.append(dates[t_end_window - 1])
    horizon_start_list.append(dates[t_end_window])
    horizon_end_list.append(dates[t_end_horizon - 1])

  return {
      'X': np.array(X_list),
      'y': np.array(y_list),
      'window_start': np.array(window_start_list),
      'window_end': np.array(window_end_list),
      'horizon_start': np.array(horizon_start_list),
      'horizon_end': np.array(horizon_end_list)}

# Ejecuta preprocesamiento para la segmentación de datos
def build_supervised_datasets(df, feature_cols = feature_cols, target_col = target_col):
  n_total_days = len(df)
  split_info = calculate_window_based_split(n_total_days)
  split_dfs = split_data(df, split_info)

  datasets = {}

  for split_name in split_names:
    datasets[split_name] = generate_windows_and_horizons(
        split_dfs[split_name],
        feature_cols,
        target_col)

  datasets['split_info'] = split_info

  return datasets

# Realiza verificaciones de sanidad
def validate_partition(datasets):
  info = datasets['split_info']

  # Validación de órden cronológico y no traslape entre conjuntos
  train_horizon_last_day = datasets['train']['horizon_end'][-1]
  val_window_first_day = datasets['val']['window_start'][0]
  val_horizon_last_day = datasets['val']['horizon_end'][-1]
  test_window_first_day = datasets['test']['window_start'][0]

  assert train_horizon_last_day < val_window_first_day, 'Posible fuga de datos'
  assert val_horizon_last_day < test_window_first_day, 'Posible fuga de datos'

  # Forma consistente de arreglos
  for split_name in split_names:
    X = datasets[split_name]['X']
    y = datasets[split_name]['y']

    assert X.shape[0] == y.shape[0] == info[split_name]['n_windows'], f'Número de muestras inconsistente en {split_name}'
    assert X.shape[1] == window_size, f'Tamaño de ventana inconsistente en {split_name}'
    assert y.shape[1] == horizon, f'Horizonte inconsistente en {split_name}'

  print('No se detecta fuga de datos')

  for split_name in split_names:
    print(
        f"{split_name}:\n"
        f"Ventanas: {info[split_name]['n_windows']}\n"
        f"Días usados: {info[split_name]['n_days']}\n")

### 3.3 Transformación de variables con tendencia

In [ ]:
# Crea copia del conjunto de datos
df_data_encoding = df_data.copy()

# Transforma variables de generación a tasas de crecimiento logarítmico (log-diff)
# log_diff(t) = log(x_t + eps) - log(x_{t-1} + eps)
for col in generation_cols:
  df_data_encoding[col] = np.log(df_data_encoding[col] + log_diff_eps).diff()

# Elimina primer día (valor nulo generado por diff())
df_data_encoding = df_data_encoding.iloc[1: ]

print(f'Días disponibles para codificación visual: {len(df_data_encoding): ,}')

# Descripción del conjunto de datos
df_data_encoding[generation_cols].describe()

### 3.4 Esquema de ventanas deslizantes

In [ ]:
# Construye los pares (ventana, horizonte) para entrenamiento, validación y prueba
df_data_split_encoding = build_supervised_datasets(df_data_encoding, feature_cols, target_col)

# Realiza verificaciones de sanidad
validate_partition(df_data_split_encoding)

In [ ]:
for split_name in split_names:
  print(
      f"{split_name}:\n"
      f'Ventana "total": {pd.Timestamp(df_data_split_encoding[split_name]['window_start'][0]).strftime('%Y-%m-%d')} - {pd.Timestamp(df_data_split_encoding[split_name]['window_end'][-1]).strftime('%Y-%m-%d')}\n'
      f'Horizonte "final": {pd.Timestamp(df_data_split_encoding[split_name]['horizon_start'][-1]).strftime('%Y-%m-%d')} - {pd.Timestamp(df_data_split_encoding[split_name]['horizon_end'][-1]).strftime('%Y-%m-%d')}\n')

## 4. Evaluación y pronóstico

### 4.1 Definición de funciones

In [ ]:
# Anclas para revertir transformación log-diff
# Cada ancla es el valor original de la variable objetivo del día anterior al inicio del horizonte
def get_target_anchors(split_name):
  window_end_dates = pd.DatetimeIndex(df_data_split_encoding[split_name]['window_end'])

  return df_data.loc[window_end_dates, target_col].to_numpy()

# Reconstruye malla de fechas del horizonte de pronóstico
def get_horizon_dates(split_name = 'test', datasets = None):
  datasets = datasets if datasets is not None else df_data_split_encoding
  horizon_start = pd.to_datetime(datasets[split_name]['horizon_start'])

  return np.array([horizon_start + pd.Timedelta(days = d) for d in range(horizon)]).T

# Revierte la transformación log-diff
# log(y_t) = log(ancla) + suma acumulada de las tasas log-diff del horizonte
def inverse_log_diff_target(y_log_diff, anchors, eps = log_diff_eps):
  log_anchors = np.log(anchors + eps).reshape(-1, 1)
  log_levels = log_anchors + np.cumsum(y_log_diff, axis = 1)

  return np.exp(log_levels) - eps

# Aplica la reversión de la transformación log-diff
def to_original_scale(y_values, anchors):
  if target_col in generation_cols:
    return inverse_log_diff_target(y_values, anchors)

  return np.asarray(y_values)

In [ ]:
# Construye instancia del optimizador indicado
def get_optimizer(optimizer_name, learning_rate):
  optimizers = {
      'adam': Adam(learning_rate = learning_rate),
      'rmsprop': RMSprop(learning_rate = learning_rate),
      'sgd': SGD(learning_rate = learning_rate, momentum = 0.9)}

  return optimizers[optimizer_name]

# Define métricas de evaluación
def forecast_metrics(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true.ravel(), y_pred.ravel())
    rmse = np.sqrt(mean_squared_error(y_true.ravel(), y_pred.ravel()))
    mape = np.mean(np.abs((y_true.ravel() - y_pred.ravel()) / y_true.ravel())) * 100

    print('Métricas de evaluación:')
    print(f'MAE: {round(mae, 3)} {generation_unit}')
    print(f'RMSE: {round(rmse, 3)} {generation_unit}')
    print(f'MAPE: {round(mape, 3)}%')

    return {'Modelo': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

# Callbacks para controlar proceso de entrenamiento
# Genera lista de callbacks por modelo
def build_callbacks(model_name):
    checkpoint = ModelCheckpoint(
        filepath = f'{base_path}/02_Modelos/{model_name}.keras',
        monitor = 'val_loss',
        save_best_only = True,
        verbose = 0)

    early_stop = EarlyStopping(
        monitor = 'val_loss',
        patience = EARLY_STOP_PATIENCE,
        restore_best_weights = True,
        verbose = 1)

    reduce_lr = ReduceLROnPlateau(
        monitor = 'val_loss',
        factor = 0.5,
        patience = REDUCE_LR_PATIENCE,
        min_lr = 1e-6,
        verbose = 1)

    return [early_stop, reduce_lr]

# Grafica historia de entrenamiento
def training_history_plot(model_history):
  train_loss = model_history.history['loss']
  val_loss = model_history.history['val_loss']
  train_mape = model_history.history['mape']
  val_mape = model_history.history['val_mape']

  plt.figure(figsize = (10, 4))

  plt.subplot(1, 2, 1)
  plt.plot(model_history.epoch, train_loss, 'r', label = 'Pérdida en entrenamiento')
  plt.plot(model_history.epoch, val_loss, 'b', label = 'Pérdida en validación')
  plt.title('Pérdida')
  plt.xlabel('Época')
  plt.ylabel('Pérdida')
  plt.legend()
  plt.grid(True)

  plt.subplot(1, 2, 2)
  plt.plot(model_history.epoch, train_mape, 'r', label = 'MAPE en entrenamiento')
  plt.plot(model_history.epoch, val_mape, 'b', label = 'MAPE en validación')
  plt.title('MAPE')
  plt.xlabel('Época')
  plt.ylabel('MAPE')
  plt.legend()
  plt.grid(True)

  plt.show()

# Grafica predicciones
def predictions_plot(model_name, y_test, y_pred, n_points = 200):
  y_test_flat = y_test.ravel()
  y_pred_flat = y_pred.ravel()

  plt.figure(figsize = (10, 4))

  plt.plot(y_test_flat[: n_points], label = 'Real')
  plt.plot(y_pred_flat[: n_points], label = 'Predicción')

  plt.xlabel('Días')
  plt.ylabel(f'{generation_unit}')
  plt.title(f'Modelo {model_name} - Predicción vs Real')
  plt.legend()
  plt.show()

In [ ]:
# Genera bloque de calibración (primeras n ventanas del conjunto de prueba)
# Genera bloque de evaluación (número de ventanas restantes del conjunto de prueba)
def split_calibration_test(n_test_windows, n_cal = N_CALIBRATION,
                           coverage = BAND_CONFIDENCE_A):
  # Calcula número mínimo de ventanas requeridas
  min_required = int(np.ceil(coverage / (1 - coverage)))

  assert n_cal >= min_required, (
      f'Calibración insuficiente: se requieren al menos {min_required} ventanas para una confianza nominal de {coverage}')
  assert n_cal < n_test_windows, (
      f'El bloque de calibración ({n_cal}) no deja ventanas de evaluación (disponibles: {n_test_windows})')

  cal_idx = np.arange(0, n_cal)
  eval_idx = np.arange(n_cal, n_test_windows)

  print(f'Calibración: ventanas 0 a {n_cal - 1} ({n_cal} ventanas, {n_cal * horizon} días de horizonte)')
  print(f'Evaluación: ventanas {n_cal} a {n_test_windows - 1} ({len(eval_idx)} ventanas, {len(eval_idx) * horizon} días de horizonte)')
  print(f'Resolución de la cobertura por día: {round(100 / len(eval_idx), 2)}%')

  return cal_idx, eval_idx

# Cuantil conformal del score de no conformidad (error absoluto en escala original)
# Calculado por día del horizonte de pronóstico
# Garantiza cobertura marginal >= cobertura nominal bajo intercambiabilidad
def conformal_quantile(scores, coverage = BAND_CONFIDENCE_A):
  n = scores.shape[0]

  # Corrección de muestra finita
  level = min(np.ceil((n + 1) * coverage) / n, 1.0)

  return np.quantile(scores, level, axis = 0, method = 'higher')

# Calibra los márgenes de la banda conformal
def calibrate_conformal_bands(y_cal, pred_cal,
                              coverages = (BAND_CONFIDENCE_B, BAND_CONFIDENCE_A)):
  # Calcula score de no conformidad (error absoluto en escala original) en el bloque de calibración
  scores = np.abs(y_cal - pred_cal)

  # Crea diccionario de cuantiles calculados
  quantiles = {coverage: conformal_quantile(scores, coverage) for coverage in coverages}

  table = pd.DataFrame({'Día': np.arange(1, scores.shape[1] + 1)})

  for coverage, q in quantiles.items():
    table[f'q{int(coverage * 100)}'] = q

  print(f'\nMárgenes conformales por día del horizonte ({generation_unit}):')
  display(table.round(3))

  return quantiles

# Construye banda simétrica alrededor del pronóstico
def apply_conformal_bands(pred, quantiles):
  return {coverage: {'lower': pred - q, 'upper': pred + q} for coverage, q in quantiles.items()}

# Evalúa cobertura empírica y ancho promedio por día del horizonte de pronóstico
def band_coverage_report(model_name, y_eval, pred_eval, bands):
  # Calcula residuos en el bloque de evaluación
  residuals = y_eval - pred_eval
  rows = []

  # Identifica valores predichos que caen dentro de la banda conformal
  for coverage, band in bands.items():
    inside = (y_eval >= band['lower']) & (y_eval <= band['upper'])

    for d in range(y_eval.shape[1]):
      rows.append({
          'Confianza': coverage,
          'Dia': d + 1,
          'MAE': np.abs(residuals[:, d]).mean(),
          'RMSE': np.sqrt((residuals[:, d] ** 2).mean()),
          'Cobertura': inside[:, d].mean(),
          'Ancho': (band['upper'][:, d] - band['lower'][:, d]).mean()})

  df_coverage = pd.DataFrame(rows)

  # Calcula estadísticos de cobertura
  summary = (df_coverage
             .groupby(['Confianza'], as_index = False)
             .agg(MAE = ('MAE', 'mean'), RMSE = ('RMSE', 'mean'),
                  Cobertura = ('Cobertura', 'mean'), Ancho = ('Ancho', 'mean')))

  # Calcula error de cobertura
  summary['Error_cobertura'] = (summary['Cobertura'] - summary['Confianza']).abs()

  print('\nCobertura por día del horizonte de pronóstico:')
  display(df_coverage.round(3))

  print('\nResumen de cobertura sobre el bloque de evaluación:')
  display(summary.round(3))

  return df_coverage

# Grafica predicción, valor real y bandas sobre el bloque de evaluación
# Las ventanas se concatenan cronológicamente
def prediction_bands_plot(model_name, y_eval, pred_eval, bands,
                          n_points = 210, horizon_dates = None):
  y_flat = y_eval.ravel()
  pred_flat = pred_eval.ravel()

  # Ventana de presentación
  n_points = min(n_points, len(y_flat))

  if horizon_dates is not None:
    x_axis = pd.to_datetime(np.asarray(horizon_dates).ravel())[-n_points:]
    unique_years = np.unique(x_axis.year)
    year_str = f'({unique_years[0]})' if len(unique_years) == 1 else f'({unique_years[0]}-{unique_years[-1]})'
    x_label = f'Fecha {year_str}'
  else:
    x_axis = np.arange(len(y_flat))[-n_points:]
    x_label = 'Días'

  y_flat = y_flat[-n_points:]
  pred_flat = pred_flat[-n_points:]

  # Tamaño de la figura
  plt.figure(figsize = (FIG_W, FIG_H))

  # Grafica bandas conformales
  for coverage in sorted(bands.keys(), reverse = True):
    cfg = band_configs.get(coverage, {'alpha': 0.15, 'label': f'IP {int(coverage * 100)} %'})
    plt.fill_between(
        x_axis,
        bands[coverage]['lower'].ravel()[-n_points:],
        bands[coverage]['upper'].ravel()[-n_points:],
        color = orange_base, alpha = cfg['alpha'],
        label = cfg['label'])

  # Grafica series principales con marcadores
  plt.plot(x_axis, y_flat, linewidth = 1.2, color = '#1F77B4',
           marker = 'o', markersize = 2.5, label = 'Real')
  plt.plot(x_axis, pred_flat, linewidth = 1.2, color = orange_base,
           marker = 'o', markersize = 2.5, label = 'Pronóstico')

  # Identifica valores reales fuera de la banda conformal externa (IP 90%)
  coverage_max = max(bands)
  outside = ((y_flat < bands[coverage_max]['lower'].ravel()[-n_points:]) |
             (y_flat > bands[coverage_max]['upper'].ravel()[-n_points:]))

  # Grafica valores reales fuera de la banda conformal externa
  if outside.any():
    outside_count = int(outside.sum())
    plt.scatter(np.asarray(x_axis)[outside], y_flat[outside],
                color = '#B2182B', marker = 'x', s = 30, zorder = 5,
                label = f'Fuera de IP 90 % ({outside_count}/{n_points})')

  plt.xlabel(x_label, fontsize = FS_LABEL)
  plt.ylabel(f'Pronóstico ({generation_unit})', fontsize = FS_LABEL)

  # Ajusta rango vertical dinámico
  y_min, y_max = min(y_flat.min(), pred_flat.min()), max(y_flat.max(), pred_flat.max())
  plt.ylim(y_min - 10, y_max + 10)

  # Ajuste de límites de fechas y formato
  if horizon_dates is not None:
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval = 4))
    plt.xlim(left = x_axis[0] - pd.Timedelta(days = 1), right = x_axis[-1] + pd.Timedelta(days = 1))
  else:
    plt.xlim(left = x_axis[0] - 0.5, right = x_axis[-1] + 0.5)

  # Leyenda
  handles, labels = plt.gca().get_legend_handles_labels()
  order_dict = {'Real': 0, 'Pronóstico': 1, 'IP 80 %': 2, 'IP 90 %': 3}
  ordered_handles = [None] * 5
  ordered_labels = [None] * 5

  for h, l in zip(handles, labels):
    if l in order_dict:
      idx = order_dict[l]
      ordered_handles[idx] = h
      ordered_labels[idx] = l
    elif 'Fuera' in l:
      ordered_handles[4] = h
      ordered_labels[4] = l

  final_handles = [h for h in ordered_handles if h is not None]
  final_labels = [l for l in ordered_labels if l is not None]

  # Ubicación de la leyenda
  plt.legend(final_handles, final_labels, loc = 'lower center', bbox_to_anchor = (0.5, 1.01),
             ncol = 5, frameon = False, fontsize = FS_LEGEND)

  # Rejilla
  plt.grid(True, axis = 'y', color = '#E0E0E0', linestyle = '-', linewidth = 0.5)
  plt.grid(False, axis = 'x')

  plt.tick_params(labelsize = 9)

  # Guarda la imagen con alta resolución
  filename = f'Prediccion_conformal_v1.png'
  plt.savefig(filename, dpi = 600, bbox_inches = 'tight')
  plt.show()

  # Descarga el archivo
  # files.download(filename)

# Ejecuta flujo completo de la banda conformal
# Calibra en el bloque inicial del conjunto de prueba
# Evalúa en el bloque restante del conjunto de prueba
def run_conformal_bands(model_name, y_test_original, pred_test_original,
                        n_cal = N_CALIBRATION, display_windows = BAND_WINDOWS):
  cal_idx, eval_idx = split_calibration_test(len(y_test_original), n_cal)

  # Márgenes congelados, estimados con el bloque de calibración
  quantiles = calibrate_conformal_bands(
      y_test_original[cal_idx], pred_test_original[cal_idx])

  y_eval = y_test_original[eval_idx]
  pred_eval = pred_test_original[eval_idx]

  # Banda de predicción simétrica alrededor del pronóstico en el bloque de evaluación
  bands = apply_conformal_bands(pred_eval, quantiles)

  # Crea reporte de cobertura en el bloque de evaluación
  df_coverage = band_coverage_report(model_name, y_eval, pred_eval, bands)
  horizon_dates_eval = get_horizon_dates('test')[eval_idx]

  # Grafica de cobertura
  for n_points in display_windows:
    print('\n')

    prediction_bands_plot(model_name, y_eval, pred_eval, bands,
                          n_points = n_points, horizon_dates = horizon_dates_eval)

  return {'quantiles': quantiles, 'bands': bands, 'coverage': df_coverage,
          'cal_idx': cal_idx, 'eval_idx': eval_idx}

In [ ]:
# Calcula estadísticos descriptivos de residuos (real - predicho)
# Valores atípicos se identifican como 1.5 * IQR (no asume normalidad de los residuos)
def residuals_statistics(residuals, iqr_factor = 1.5):
  residuals_flat = residuals.ravel()

  p25 = np.percentile(residuals_flat, 25)
  p50 = np.percentile(residuals_flat, 50)
  p75 = np.percentile(residuals_flat, 75)
  iqr = p75 - p25

  lower_bound = p25 - iqr_factor * iqr
  upper_bound = p75 + iqr_factor * iqr

  outliers_mask = (residuals_flat < lower_bound) | (residuals_flat > upper_bound)

  return {
      'mean': residuals_flat.mean(),
      'std': residuals_flat.std(ddof = 1),
      'min': residuals_flat.min(),
      'p25': p25,
      'p50': p50,
      'p75': p75,
      'max': residuals_flat.max(),
      'iqr': iqr,
      'lower_bound': lower_bound,
      'upper_bound': upper_bound,
      'num_outliers': int(outliers_mask.sum()),
      'pct_outliers': 100 * outliers_mask.mean(),
      'records': len(residuals_flat)}

# Grafica análisis de residuos en cuatro paneles
def residuals_plot(model_name, residuals, stats, horizon_dates = None, iqr_factor = 1.5):
  residuals_flat = residuals.ravel()
  outliers_mask = ((residuals_flat < stats['lower_bound']) |
                   (residuals_flat > stats['upper_bound']))

  # Eje temporal con fechas reales si se proporcionan (índice secuencial en caso contratio)
  if horizon_dates is not None:
    x_axis = pd.to_datetime(np.asarray(horizon_dates).ravel())
    x_label = 'Fecha'
  else:
    x_axis = np.arange(len(residuals_flat))
    x_label = 'Días'

  fig, axs = plt.subplots(2, 2, figsize = (16, 9))

  # Panel 1. Serie temporal de residuos
  axs[0, 0].plot(x_axis, residuals_flat, linewidth = 0.9, color = 'steelblue', label = 'Residuo')
  axs[0, 0].axhline(0, color = 'black', linewidth = 1)
  axs[0, 0].axhline(stats['mean'], color = 'darkorange', linestyle = '--',
                    linewidth = 1.2, label = f"Promedio ({round(stats['mean'], 2)})")
  axs[0, 0].fill_between(
      x_axis, stats['mean'] - stats['std'],
      stats['mean'] + stats['std'],
      color = 'darkorange', alpha = 0.15, label = '+/- 1 desviación estándar')

  if outliers_mask.any():
    axs[0, 0].scatter(np.asarray(x_axis)[outliers_mask], residuals_flat[outliers_mask],
                      color = 'crimson', s = 18, zorder = 5,
                      label = f"Atípicos ({stats['num_outliers']})")

  axs[0, 0].set_title('Serie de residuos (real - predicho)')
  axs[0, 0].set_xlabel(x_label)
  axs[0, 0].set_ylabel(f'Residuo ({generation_unit})')
  axs[0, 0].legend(fontsize = 8)
  axs[0, 0].grid(True, alpha = 0.3)

  if horizon_dates is not None:
    axs[0, 0].tick_params(axis = 'x', rotation = 45)

  # Panel 2. Distribución de residuos
  axs[0, 1].hist(residuals_flat, bins = 40, color = 'steelblue', edgecolor = 'white', alpha = 0.85)
  axs[0, 1].axvline(stats['p25'], color = 'seagreen', linestyle = '--',
                    linewidth = 1.2, label = f"P25 ({round(stats['p25'], 2)})")
  axs[0, 1].axvline(stats['p50'], color = 'darkorange', linestyle = '-',
                    linewidth = 1.5, label = f"P50 ({round(stats['p50'], 2)})")
  axs[0, 1].axvline(stats['p75'], color = 'seagreen', linestyle = '--',
                    linewidth = 1.2, label = f"P75 ({round(stats['p75'], 2)})")
  axs[0, 1].axvline(stats['lower_bound'], color = 'crimson', linestyle = ':', linewidth = 1.2)
  axs[0, 1].axvline(stats['upper_bound'], color = 'crimson', linestyle = ':',
                    linewidth = 1.2, label = f'Límites {iqr_factor} * IQR')

  axs[0, 1].set_title('Distribución de residuos')
  axs[0, 1].set_xlabel(f'Residuo ({generation_unit})')
  axs[0, 1].set_ylabel('Frecuencia')
  axs[0, 1].legend(fontsize = 8)
  axs[0, 1].grid(True, alpha = 0.3)

  # Panel 3. Diagrama de caja
  axs[1, 0].boxplot(residuals_flat, vert = False, whis = iqr_factor,
                    patch_artist = True,
                    boxprops = dict(facecolor = 'steelblue', alpha = 0.6),
                    medianprops = dict(color = 'darkorange', linewidth = 2),
                    flierprops = dict(marker = 'o', markersize = 4, markerfacecolor = 'crimson',
                                      markeredgecolor = 'crimson', alpha = 0.6))
  axs[1, 0].axvline(0, color = 'black', linewidth = 1, linestyle = '--')
  axs[1, 0].set_title('Diagrama de caja de residuos')
  axs[1, 0].set_xlabel(f'Residuo ({generation_unit})')
  axs[1, 0].set_yticks([])
  axs[1, 0].grid(True, alpha = 0.3)

  # Panel 4. Residuos por día del horizonte de pronóstico
  residuals_by_day = [residuals[:, d] for d in range(residuals.shape[1])]
  axs[1, 1].boxplot(residuals_by_day, whis = iqr_factor,
                    patch_artist = True,
                    boxprops = dict(facecolor = 'steelblue', alpha = 0.6),
                    medianprops = dict(color = 'darkorange', linewidth = 2),
                    flierprops = dict(marker = 'o', markersize = 3, markerfacecolor = 'crimson',
                                      markeredgecolor = 'crimson', alpha = 0.5))
  axs[1, 1].axhline(0, color = 'black', linewidth = 1, linestyle = '--')
  axs[1, 1].set_title('Residuos por día del horizonte de pronóstico')
  axs[1, 1].set_xlabel('Día')
  axs[1, 1].set_ylabel(f'Residuo ({generation_unit})')
  axs[1, 1].grid(True, alpha = 0.3)

  plt.suptitle(f'Modelo {model_name} - Análisis de residuos')
  plt.tight_layout()
  plt.show()

# Ejecuta análisis completo de residuos (cálculo, reporte y gráficas)
def residuals_analysis(model_name, y_true, y_pred, horizon_dates = None, iqr_factor = 1.5):
  residuals = np.asarray(y_true) - np.asarray(y_pred)
  stats = residuals_statistics(residuals, iqr_factor)

  print('Estadísticos de residuos:')
  print(f"Promedio: {round(stats['mean'], 2)}")
  print(f"Desviación estándar: {round(stats['std'], 2)}")
  print(f"P25: {round(stats['p25'], 2)}")
  print(f"P50 (mediana): {round(stats['p50'], 2)}")
  print(f"P75: {round(stats['p75'], 2)}")
  print(f"Rango intercuartílico: {round(stats['iqr'], 2)}")
  print(f"Límites de atípicos: "
        f"[{round(stats['lower_bound'], 2)}, {round(stats['upper_bound'], 2)}]")
  print(f"Valores atípicos: {stats['num_outliers']} de {stats['records']} "
        f"({round(stats['pct_outliers'], 2)}%)")

  print('\n')
  residuals_plot(model_name, residuals, stats, horizon_dates, iqr_factor)

  # Estadísticos por día del horizonte de pronóstico
  # Permite cuantificar la acumulación del error según el día
  by_day = pd.DataFrame({
      'Dia': np.arange(1, residuals.shape[1] + 1),
      'Promedio': residuals.mean(axis = 0),
      'Mediana': np.median(residuals, axis = 0),
      'Desv. Est.': residuals.std(axis = 0, ddof = 1),
      'MAE': np.abs(residuals).mean(axis = 0)})

  print('\n')
  print('Estadísticos por día del horizonte de pronóstico:')
  display(by_day.round(3))

  # Creating stats_df with the new keys
  stats_df_data = {
      'mean': [stats['mean']],
      'std': [stats['std']],
      'min': [stats['min']],
      'p25': [stats['p25']],
      'p50': [stats['p50']],
      'p75': [stats['p75']],
      'max': [stats['max']],
      'iqr': [stats['iqr']],
      'lower_bound': [stats['lower_bound']],
      'upper_bound': [stats['upper_bound']],
      'num_outliers': [stats['num_outliers']],
      'pct_outliers': [stats['pct_outliers']],
      'records': [stats['records']]}

  stats_df = pd.DataFrame(stats_df_data, index = [model_name]).round(3)

  return stats_df, residuals

### 4.2 Variable objetivo y anclas

In [ ]:
# Variable objetivo por conjunto de datos (en escala log-diff)
y_train = df_data_split_encoding['train']['y']
y_val = df_data_split_encoding['val']['y']
y_test = df_data_split_encoding['test']['y']

anchors_train = get_target_anchors('train')
anchors_val = get_target_anchors('val')
anchors_test = get_target_anchors('test')

print(f'\nMuestras - Entrenamiento: {len(y_train)}, Validación: {len(y_val)}, Prueba: {len(y_test)}')
print(f'Anclas - Entrenamiento: {anchors_train.shape}, Validación: {anchors_val.shape}, Prueba: {anchors_test.shape}')

## 5. Modelos de referencia

Establecen el punto de comparación frente al cual se evalúa el aporte de la codificación visual. Procesan la ventana temporal de forma directa.

La rama secuencial definida en esta sección será la misma que emplearán los modelos multimodales, de modo que el modelo de referencia y la rama secuencial del modelo multimodal correspondiente serán, por construcción, la misma red.

### 5.1 Importación de librerías específicas

In [ ]:
import holidays as holidays_pkg # Días festivos

### 5.2 Definición de variables

In [ ]:
# Variables base de la rama secuencial
seq_base_cols = ['generacion_hidraulica', 'generacion_gas', 'generacion_carbon', 'precio_gas', 'precio_carbon', 'ppp_bolsa', 'y']
# seq_base_cols = ['y']

# Establece el uso o no de ingeniería de características en la rama secuencial
use_calendar_features = True
# use_calendar_features = False

# Variables de calendario a generar (deterministas y conocidas)
calendar_features = ['dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'es_fin_semana', 'es_festivo']
# calendar_features = ['es_fin_semana', 'es_festivo']

# Variables sin escalamiento estadístico (binarias o ya acotadas en el rango [-1, 1])
no_scale_features = calendar_features if use_calendar_features else []

# Método de escalamiento de la rama secuencial
seq_scaling_method = 'robust'

seq_branch_type = 'LSTM' # Tipo de rama secuencial: 'LSTM', 'GRU', 'BiLSTM' o 'CNN-LSTM'
seq_units = 64 # Unidades de la capa recurrente
seq_layers = 2 # Número de capas recurrentes apiladas
seq_dropout_rate = 0.2 # Tasa de dropout de la rama secuencial
seq_conv_filters = 32 # Filtros de la Conv1D (solo para 'CNN-LSTM')
seq_conv_kernel = 3 # Tamaño de kernel de la Conv1D (solo para 'CNN-LSTM')

### 5.3 Definición de funciones

In [ ]:
# Genera variables de calendario
# Emplea df_data_encoding (no df_data) para preservar la alineación
def build_calendar_features(df, use_calendar = None):
  df_out = df.copy()

  # Permite prescindir de la ingeniería de características
  if not use_calendar_features:
    return df_out

  dow = df_out.index.dayofweek # 0 = lunes... 6 = domingo
  doy = df_out.index.dayofyear

  # Genera lista de días festivos en Colombia
  co_holidays = holidays_pkg.Colombia(
      years = range(df_out.index.year.min(), df_out.index.year.max() + 1))

  # Codificación cíclica (domingo a lunes)
  df_out['dow_sin'] = np.sin(2 * np.pi * dow / 7)
  df_out['dow_cos'] = np.cos(2 * np.pi * dow / 7)
  df_out['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
  df_out['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)

  # Identificación de días fin de semana
  df_out['es_fin_semana'] = (dow >= 5).astype(int)

  # Identificación de días festivos
  df_out['es_festivo'] = np.array([d in co_holidays for d in df_out.index.date]).astype(int)

  return df_out

# Ajusta escaladores de la rama secuencial (con conjunto de datos de entrenamiento)
def fit_sequence_scalers(train_df, seq_feature_cols, scaling_method = seq_scaling_method,
                         no_scale = no_scale_features):
  feature_scalers = {}

  for col in seq_feature_cols:
    if col in no_scale:
      feature_scalers[col] = None
      continue

    if scaling_method == 'robust':
      scaler = RobustScaler()
    elif scaling_method == 'standard':
      scaler = StandardScaler()
    elif scaling_method == 'minmax':
      scaler = MinMaxScaler()
    else:
      raise ValueError(f'Error. Método de escalamiento incorrecto {scaling_method}')

    feature_scalers[col] = scaler.fit(train_df[[col]])

  return feature_scalers

# Aplica escaladores a las ventanas de la rama secuencial
def apply_sequence_scalers(X, seq_feature_cols, scalers):
  X_scaled = np.empty_like(X, dtype = float)

  for i, var in enumerate(seq_feature_cols):
    if scalers[var] is None:
      X_scaled[:, :, i] = X[:, :, i]
      continue

    original_shape = X[:, :, i].shape
    scaled = scalers[var].transform(X[:, :, i].reshape(-1, 1))
    X_scaled[:, :, i] = scaled.reshape(original_shape)

  return X_scaled

In [ ]:
# Construye rama secuencial
# Procesa la ventana temporal cruda
def build_sequential_branch(input_shape, branch_type = seq_branch_type, units = seq_units,
                            n_layers = seq_layers, dropout_rate = seq_dropout_rate,
                            conv_filters = seq_conv_filters, conv_kernel = seq_conv_kernel):
  inputs = layers.Input(shape = input_shape, name = 'input_sequential')
  x = inputs

  if branch_type == 'CNN-LSTM':
    # Extracción de patrones locales antes del modelado de la dependencia temporal
    x = layers.Conv1D(filters = conv_filters, kernel_size = conv_kernel, padding = 'causal',
                      activation = 'relu', name = 'seq_conv1d')(x)
    x = layers.MaxPooling1D(pool_size = 2, name = 'seq_pool')(x)

  for i in range(n_layers):
    is_last = (i == n_layers - 1)
    return_sequences = not is_last # La última capa colapsa la secuencia

    if branch_type in ('LSTM', 'CNN-LSTM'):
      x = layers.LSTM(units, return_sequences = return_sequences, name = f'seq_lstm_{i + 1}')(x)
    elif branch_type == 'GRU':
      x = layers.GRU(units, return_sequences = return_sequences, name = f'seq_gru_{i + 1}')(x)
    elif branch_type == 'BiLSTM':
      x = layers.Bidirectional(layers.LSTM(units, return_sequences = return_sequences), name = f'seq_bilstm_{i + 1}')(x)
    else:
      raise ValueError(f'Error. Tipo de rama secuencial incorrecto {branch_type}')

    x = layers.Dropout(dropout_rate, name = f'seq_dropout_{i + 1}')(x)

  return inputs, x

# Construye un modelo de referencia: rama secuencial seguida de la cabeza de pronóstico
def build_reference_model(seq_input_shape, horizon, branch_type = seq_branch_type,
                          units = seq_units, n_layers = seq_layers,
                          branch_dropout_rate = seq_dropout_rate,
                          dense_units = (64, ), dropout_rate = 0.3,
                          optimizer_name = 'adam', learning_rate = 1e-3):
  inputs, x = build_sequential_branch(
      seq_input_shape, branch_type = branch_type,
      units = units, n_layers = n_layers,
      dropout_rate = branch_dropout_rate)

  for j, dense_size in enumerate(dense_units):
    x = layers.Dense(dense_size, activation = 'relu', name = f'head_dense_{j + 1}')(x)
    x = layers.Dropout(dropout_rate, name = f'head_dropout_{j + 1}')(x)

  # Capa de salida
  outputs = layers.Dense(horizon, activation = 'linear', name = 'forecast_output')(x)

  # Construye el modelo
  branch_id = branch_type.lower().replace('-', '_')
  model = models.Model(
      inputs = inputs,
      outputs = outputs,
      name = f'referencia_{branch_id}')

  # Compila el modelo
  model.compile(
      optimizer = get_optimizer(optimizer_name, learning_rate),
      loss = 'mae',
      metrics = [keras.metrics.RootMeanSquaredError(), 'mape'])

  return model

### 5.4 Preparación de tensores de entrada

In [ ]:
# Genera variables de calendario
df_data_sequential = build_calendar_features(df_data_encoding)

# Variables predictoras de la rama secuencial
seq_feature_cols = seq_base_cols + (calendar_features if use_calendar_features else [])

print(f'Variables de la rama secuencial ({len(seq_feature_cols)}): {seq_feature_cols}')

if use_calendar_features:
  print(f'Días festivos identificados: {int(df_data_sequential["es_festivo"].sum())}')
else:
  print('Ingeniería de características desactivada')

# La generación de variables de calendario no debe alterar el número de filas
assert len(df_data_sequential) == len(df_data_encoding), 'Desalineación de filas'

df_data_sequential[seq_feature_cols].head(10)

In [ ]:
# Construye los pares (ventana, horizonte) de la rama secuencial
df_data_split_sequential = build_supervised_datasets(df_data_sequential, seq_feature_cols, target_col)

# Verifica alineación de ventanas y variable objetivo con la segmentación común
# Deben coincidir para que las muestras secuenciales y las de imagen se correspondan una a una
for split_name in split_names:
  n_seq = df_data_split_sequential[split_name]['X'].shape[0]
  n_base = df_data_split_encoding[split_name]['X'].shape[0]

  assert n_seq == n_base, f'Ventanas desalineadas en {split_name}: {n_seq} vs {n_base}'
  assert np.array_equal(df_data_split_sequential[split_name]['y'],
                        df_data_split_encoding[split_name]['y']), \
      f'Variable objetivo distinta en {split_name}'

print('Ventanas alineadas con la segmentación común')

for split_name in split_names:
  print(f'{split_name}: {df_data_split_sequential[split_name]["X"].shape}')

In [ ]:
# Ajusta escaladores de la rama secuencial (con datos de entrenamiento)
split_info_seq = df_data_split_sequential['split_info']

train_df_sequential = df_data_sequential.iloc[
    split_info_seq['train']['start_day']: split_info_seq['train']['end_day'] + 1]

seq_scalers = fit_sequence_scalers(train_df_sequential, seq_feature_cols)

# Aplica escaladores a cada conjunto de datos
X_seq_train = apply_sequence_scalers(df_data_split_sequential['train']['X'], seq_feature_cols, seq_scalers)
X_seq_val = apply_sequence_scalers(df_data_split_sequential['val']['X'], seq_feature_cols, seq_scalers)
X_seq_test = apply_sequence_scalers(df_data_split_sequential['test']['X'], seq_feature_cols, seq_scalers)

# Forma de entrada de la rama secuencial: (ventana, n_variables)
seq_input_shape = X_seq_train.shape[1:]

print(f'Variables sin escalamiento estadístico: '
      f'{[c for c, s in seq_scalers.items() if s is None]}')
print(f'Forma de entrada de la rama secuencial: {seq_input_shape}')

### 5.5 Confección de modelos

In [ ]:
# Ejecuta el flujo completo para una configuración de modelo de referencia
def run_reference_experiment(
    seq_input_shape, X_seq_train, y_train,
    X_seq_val, y_val, X_seq_test, y_test,
    anchors_test, model_name = 'Referencia',
    branch_type = seq_branch_type, units = seq_units,
    n_layers = seq_layers, branch_dropout_rate = seq_dropout_rate,
    dense_units = (64, ), dropout_rate = 0.3,
    optimizer_name = 'adam', learning_rate = 1e-3,
    epochs = EPOCHS, batch_size = BATCH_SIZE):
  keras.utils.set_random_seed(SEED) # Garantiza reproducibilidad

  # Confección del modelo
  model = build_reference_model(
      seq_input_shape, horizon = horizon,
      branch_type = branch_type, units = units,
      n_layers = n_layers, branch_dropout_rate = branch_dropout_rate,
      dense_units = dense_units, dropout_rate = dropout_rate,
      optimizer_name = optimizer_name, learning_rate = learning_rate)

  # Ajuste (entrenamiento) del modelo
  print(f'Entrenamiento ({model_name})')
  history = model.fit(
      X_seq_train, y_train,
      validation_data = (X_seq_val, y_val),
      epochs = epochs,
      batch_size = batch_size,
      callbacks = build_callbacks(model_name),
      verbose = 1)

  # Historial de entrenamiento
  print('\n')
  training_history_plot(history)

  # Predicción sobre conjunto de prueba (en escala log-diff)
  pred_test = model.predict(X_seq_test, verbose = 0)

  # Revierte transformación log-diff de la variable objetivo
  y_test_original = to_original_scale(y_test, anchors_test)
  pred_test_original = to_original_scale(pred_test, anchors_test)

  # Métricas de evaluación
  print('\n')
  results = dict(forecast_metrics(model_name, y_test_original, pred_test_original))

  # Banda de predicción conformal
  run_conformal_bands(
      model_name,
      y_test_original,
      pred_test_original)

  # Análisis de residuos (real - predicho)
  print('\n')
  residuals_analysis(
      model_name,
      y_test_original,
      pred_test_original,
      horizon_dates = get_horizon_dates('test'))

  return model

### 5.6 Entrenamiento y validación

In [ ]:
# Arquitectura de referencia seleccionada
seq_branch_type = 'BiLSTM' # 'LSTM', 'GRU', 'BiLSTM' o 'CNN-LSTM'
print(f'Arquitectura de referencia seleccionada: {seq_branch_type}')

In [ ]:
# Ejecuta el flujo completo para la configuración de modelo de referencia

if 'model_reference' in globals():
  del model_reference

keras.backend.clear_session() # Libera el grafo y reinicia los contadores de nombres de capas
gc.collect()

model_reference = run_reference_experiment(
    seq_input_shape,
    X_seq_train, y_train,
    X_seq_val, y_val,
    X_seq_test, y_test,
    anchors_test,
    model_name = f'Referencia_{seq_branch_type}',
    branch_type = seq_branch_type)

## 6. Codificación visual de series temporales

### 6.1 Importación de librerías específicas

In [ ]:
# Importación de librerías específicas
import pyts
import pywt # Wavelet (CWT / Escalograma)

from pyts.image import GramianAngularField
from pyts.image import MarkovTransitionField
from pyts.image import RecurrencePlot

from scipy.signal import stft as scipy_stft # STFT / Espectrograma

### 6.2 Escalamiento de datos

In [ ]:
# Técnicas de codificación visual de series temporales
techniques = ['GASF', 'GADF', 'MTF']
# techniques = ['GASF', 'GADF', 'MTF', 'RP', 'CWT', 'STFT']

# Método de escalamiento por técnica de codificación
scaling_method_by_technique = {
    'GASF': 'minmax_symmetric', # Rango [-1, 1]
    'GADF': 'minmax_symmetric', # Rango [-1, 1]
    'MTF': 'quantile_bins', # bins discretos por cuantiles
    'RP': 'zscore', # Estandarización para distancias en espacio de fase
    'CWT': 'zscore', # Estandarización previa a la Wavelet
    'STFT': 'zscore'} # Estandarización previa a la STFT

# Parámetros para métodos de escalamiento
n_quantile_bins = 8 # Número de bins para MTF
minmax_margin = 0.05 # Margen de seguridad del 5% sobre el rango observado en entrenamiento

In [ ]:
# Ajusta escalador min-max simétrico hacia [-1, 1]
# Agrega margen de seguridad que expande el rango observado en entrenamiento
# Reduce los valores de validación y prueba que están fuera del rango visto en entrenamiento
def fit_minmax_symmetric_scaler(train_values, margin = minmax_margin):
    raw_min = float(np.min(train_values))
    raw_max = float(np.max(train_values))
    span = raw_max - raw_min

    return {
        'data_min': raw_min - margin * span,
        'data_max': raw_max + margin * span}

# Aplica escalador min-max simétrico
# Los valores fuera de min y max son recortados a -1/1 (clip = True)
def apply_minmax_symmetric_scaler(values, scaler_params, clip = True):
  data_min = scaler_params['data_min']
  data_max = scaler_params['data_max']

  scaled = 2 * ((values - data_min) / (data_max - data_min)) - 1

  if clip:
    scaled = np.clip(scaled, -1.0, 1.0)

  return scaled

# Ajusta escalador de estandarización z-score (media/desviación estándar)
def fit_zscore_scaler(train_values):
  return {
      'mean': float(np.mean(train_values)),
      'std': float(np.std(train_values)) or 1e-8} # Evita división por cero

# Aplica estandarización z-score
def apply_zscore_scaler(values, scaler_params):
  return (values - scaler_params['mean']) / scaler_params['std']

# Calcula bordes de los bins de cuantiles
def fit_quantile_bin_edges(train_values, n_bins = n_quantile_bins):
  quantiles = np.linspace(0, 1, n_bins + 1)
  edges = np.quantile(train_values, quantiles)
  edges = np.unique(edges) # Se eliminan bordes duplicados, por si hay bins degenerados

  return edges

# Discretiza un arreglo de valores en los bins definidos
# Valores fuera del rango de entrenamiento se asignan al primer o último bin (clip implícito)
def apply_quantile_bins(values, bin_edges):
  n_bins = len(bin_edges) - 1
  bin_idx = np.digitize(values, bin_edges[1: -1])

  return np.clip(bin_idx, 0, n_bins - 1)

# Ajusta escalador de acuerdo con la técnica de codificación indicada
def fit_scaler(train_values, technique, n_bins = n_quantile_bins,
               margin = minmax_margin):
  method = scaling_method_by_technique[technique]

  if method == 'minmax_symmetric':
    return fit_minmax_symmetric_scaler(train_values, margin)

  elif method == 'quantile_bins':
    edges = fit_quantile_bin_edges(train_values, n_bins)

    return {'bin_edges': edges}
  else:
    return fit_zscore_scaler(train_values)

# Aplica escalador ajustado
def apply_scaler(values, technique, scaler_params):
  method = scaling_method_by_technique[technique]

  if method == 'minmax_symmetric':
    return apply_minmax_symmetric_scaler(values, scaler_params)
  elif method == 'quantile_bins':
    return apply_quantile_bins(values, scaler_params['bin_edges'])
  else:
    return apply_zscore_scaler(values, scaler_params)

# Ajusta escaladores empleando el conjunto de datos de entrenamiento
# Ajusta un escalador por cada combinación variable - técnica
def fit_all_scalers(train_df, feature_cols = feature_cols, techniques = techniques,
                    n_bins = n_quantile_bins, margin = minmax_margin):
  scalers = {}

  for var in feature_cols:
    scalers[var] = {}
    train_values = train_df[var].to_numpy()

    for technique in techniques:
      scalers[var][technique] = fit_scaler(
          train_values,
          technique,
          n_bins,
          margin)

  return scalers

# Aplica escaladores ajustados a un arreglo de ventanas
def apply_scalers_to_dataset(X, scalers, feature_cols = feature_cols, techniques = techniques):
  scaled = {}

  for idx, var in enumerate(feature_cols):
    scaled[var] = {}
    raw_var_windows = X[:, :, idx] # (n_samples, window_size)

    for technique in techniques:
      scaled[var][technique] = apply_scaler(raw_var_windows, technique, scalers[var][technique])

  return scaled

# Diagnóstico de cobertura del rango de valores validación y prueba frente a entrenamiento
def Out_of_range_report(eval_df, scalers, feature_cols = feature_cols, techniques = techniques):
  rows = []

  # Porcentaje de valores que caen fuera del rango observado (y ajustado) en entrenamiento
  for var in feature_cols:
    values = eval_df[var].to_numpy()

    for technique in techniques:
      method = scaling_method_by_technique[technique]
      params = scalers[var][technique]

      if method == 'minmax_symmetric':
        out_of_range = (values < params['data_min']) | (values > params['data_max'])
        pct = 100 * out_of_range.mean()
      elif method == 'quantile_bins':
        edges = params['bin_edges']
        out_of_range = (values < edges[0]) | (values > edges[-1])
        pct = 100 * out_of_range.mean()
      else:
        pct = np.nan # No aplica para método z-score

      rows.append({'variable': var, 'technique': technique, 'pct_fuera_rango': round(pct, 2) if not np.isnan(pct) else None})

  return pd.DataFrame(rows)

In [ ]:
# Segmenta datos en conjuntos de entrenamiento, validación y prueba
train_df_raw_encoding = df_data_encoding.iloc[
    df_data_split_encoding['split_info']['train']['start_day']: df_data_split_encoding['split_info']['train']['end_day'] + 1]

val_df_raw_encoding = df_data_encoding.iloc[
    df_data_split_encoding['split_info']['val']['start_day']: df_data_split_encoding['split_info']['val']['end_day'] + 1]

test_df_raw_encoding = df_data_encoding.iloc[
    df_data_split_encoding['split_info']['test']['start_day']: df_data_split_encoding['split_info']['test']['end_day'] + 1]

# Ajusta escaladores con datos de entrenamiento
scalers = fit_all_scalers(train_df_raw_encoding, feature_cols, techniques, n_quantile_bins, minmax_margin)

print('Técnicas ajustadas por variable:')
for var, techs in scalers.items():
  print(f'{var}: {list(techs.keys())}')

In [ ]:
# Aplica escaladores ajustados a ventanas de datos de entrenamiento, validación y prueba
scaled_windows = {}

for split_name in split_names:
  scaled_windows[split_name] = apply_scalers_to_dataset(
      df_data_split_encoding[split_name]['X'], scalers, feature_cols, techniques)

# Diagnóstico de cobertura
# Presenta valores de validación y prueba que están fuera del rango visto en entrenamiento
print('Conjunto de datos de validación')
display(Out_of_range_report(val_df_raw_encoding, scalers, feature_cols, techniques))

print('Conjunto de datos de prueba')
display(Out_of_range_report(test_df_raw_encoding, scalers, feature_cols, techniques))

### 6.3 Representaciones visuales

In [ ]:
# Configuraciones de técnicas de codificación visual

# Markov Transition Fields
mtf_backend = 'global_train'

# Recurrence Plots
rp_embedding_dim = 1 # Mantiene la imagen en (window_size, window_size)
rp_time_delay = 1
rp_threshold = None # RP continuo (recomendado para CNN)

# Escalogramas (CWT)
cwt_wavelet = 'morl' # Wavelet de Morlet
cwt_n_scales = window_size # Igual al tamaño de ventana para mantener imágenes cuadradas comparables

# Espectrogramas (STFT)
stft_fs = 1.0 # Tasa de muestreo. Una muestra/día
stft_window_type = 'hann' # Tipo de ventana
stft_nperseg = 20 # Número de muestras por segmento

# Número de muestras que se solapan entre segmentos consecutivos
# Un valor alto genera un espectrograma más suave y una mejor detección de eventos transitorios
stft_noverlap = 17

# Número de muestras utilizadas para calcular la DFT de cada segmento
# Si es mayor a nperseg, los datos del segmento se rellenan con ceros hasta alcanzar nfft puntos
# El relleno con ceros no añade nueva información
# El relleno hace que los intervalos de frecuencia parezcan más densos
# El relleno permite, potencialmente, revelar detalles más precisos en el dominio de la frecuencia
stft_nfft = 40

In [ ]:
# Construye imágenes GASF para un lote de ventanas ya escaladas a [-1, 1]
def build_gasf_images(scaled_windows_2d):
  window_size = scaled_windows_2d.shape[1]

  # sample_range = None indica que los datos ya están en el rango [-1, 1]
  transformer = GramianAngularField(
        image_size = window_size, method = 'summation', sample_range = None)

  return transformer.transform(scaled_windows_2d)

# Construye imágenes GADF para un lote de ventanas ya escaladas a [-1, 1]
def build_gadf_images(scaled_windows_2d):
  window_size = scaled_windows_2d.shape[1]

  # sample_range = None indica que los datos ya están en el rango [-1, 1]
  transformer = GramianAngularField(
        image_size = window_size, method = 'difference', sample_range = None)

  return transformer.transform(scaled_windows_2d)

# Ajusta la matriz de transición de Markov de orden 1, W (n_bins x n_bins)
# Emplea todos los datos de entrenamiento
def fit_transition_matrix(train_raw_values, bin_edges):
  # Asigna una distribucion uniforme (fallback) si el estado nunca aparece como origen en entrenamiento
  n_bins = len(bin_edges) - 1
  binned_sequence = apply_quantile_bins(train_raw_values, bin_edges)
  counts = np.zeros((n_bins, n_bins), dtype = float)

  for t in range(len(binned_sequence) - 1):
    i, j = binned_sequence[t], binned_sequence[t + 1]
    counts[i, j] += 1

  row_sums = counts.sum(axis = 1, keepdims = True)
  W = np.divide(counts, row_sums, out = np.full_like(counts, 1.0 / n_bins), where = row_sums > 0)

  return W

# Ajusta la matriz de transición de MTF para cada variable (mtf_backend = 'global_train')
def fit_all_transition_matrices(train_df, scalers, feature_cols = feature_cols):
  transition_matrices = {}

  # La matriz de transición solo aplica a MTF
  if 'MTF' not in techniques:
    print('No se ajustan matrices de transición de Markov')

    return transition_matrices

  for var in feature_cols:
    bin_edges = scalers[var]['MTF']['bin_edges']
    train_raw_values = train_df[var].to_numpy()
    transition_matrices[var] = fit_transition_matrix(train_raw_values, bin_edges)

  return transition_matrices

# Backend 'global_train': expande, para un lote de ventanas ya discretizadas
def build_mtf_images_global_train(binned_windows_2d, transition_matrix):
  binned = binned_windows_2d.astype(int)

  # Indexado avanzado vectorizado: para cada muestra k, W[binned[k, i], binned[k, j]]
  return transition_matrix[binned[:, :, None], binned[:, None, :]]

# Backend 'pyts': usa librería pyts
# Calcula bordes de cuantiles y matriz de transición por ventana
def build_mtf_images_pyts(raw_windows_2d, n_bins = n_quantile_bins):
  window_size = raw_windows_2d.shape[1]

  transformer = MarkovTransitionField(
        image_size = window_size, n_bins = n_bins, strategy = 'quantile')

  return transformer.transform(raw_windows_2d)

# Construye imágenes RP para un lote de ventanas ya estandarizadas (z-score)
def build_rp_images(scaled_windows_2d, dimension = rp_embedding_dim,
                    time_delay = rp_time_delay, threshold = rp_threshold):
  # threshold = None retorna el RP continuo (recomendado para CNN)
  transformer = RecurrencePlot(dimension = dimension, time_delay = time_delay, threshold = threshold)

  return transformer.transform(scaled_windows_2d)

# Construye escalogramas (magnitud de la CWT) para un lote de ventanas ya estandarizadas (z-score)
def build_cwt_images(scaled_windows_2d, wavelet = cwt_wavelet, n_scales = cwt_n_scales):
  scales = np.arange(1, n_scales + 1)
  coeffs, _freqs = pywt.cwt(scaled_windows_2d, scales, wavelet, axis = -1)

  # pywt retorna (n_scales, n_samples, window_size). Reordena a (n_samples, n_scales, window_size)
  coeffs = np.transpose(coeffs, (1, 0, 2))

  return np.abs(coeffs)

# Construye espectrogramas (magnitud de la STFT) para un lote de ventanas ya estandarizadas (z-score)
def build_stft_images(scaled_windows_2d, fs = stft_fs, window_type = stft_window_type,
                      nperseg = stft_nperseg, noverlap = stft_noverlap, nfft = stft_nfft):
  _f, _t, Zxx = scipy_stft(
      scaled_windows_2d, fs = fs, window = window_type,
      nperseg = nperseg, noverlap = noverlap, nfft = nfft, axis = -1)

  return np.abs(Zxx)

# Construye las imágenes de codificación visual de un lote de ventanas por técnica
def build_images(technique, scaled_windows_2d, transition_matrix = None, mtf_backend = mtf_backend):
  if technique == 'GASF':
    return build_gasf_images(scaled_windows_2d)
  elif technique == 'GADF':
    return build_gadf_images(scaled_windows_2d)
  elif technique == 'MTF':
    if mtf_backend == 'global_train':
      if transition_matrix is None:
        raise ValueError('Se requiere matriz de transición')

      return build_mtf_images_global_train(scaled_windows_2d, transition_matrix)
    else:
      return build_mtf_images_pyts(scaled_windows_2d)
  elif technique == 'RP':
    return build_rp_images(scaled_windows_2d)
  elif technique == 'CWT':
    return build_cwt_images(scaled_windows_2d)
  elif technique == 'STFT':
    return build_stft_images(scaled_windows_2d)
  else:
    raise ValueError(f'Técnica de codificación desconocida: {technique}')

# Construye las imágenes de codificación visual para un conjunto completo de datos
def encode_dataset(scaled_windows, transition_matrices, feature_cols = feature_cols,
                   techniques = techniques, mtf_backend = mtf_backend, raw_windows_for_mtf = None):
  images = {}

  for var in feature_cols:
    images[var] = {}

    for technique in techniques:
      if technique == 'MTF' and mtf_backend == 'pyts':
        data = raw_windows_for_mtf[var]
      else:
        data = scaled_windows[var][technique]

      tm = transition_matrices.get(var) if technique == 'MTF' else None
      images[var][technique] = build_images(technique, data, tm, mtf_backend)

  return images

In [ ]:
# Ajusta matriz de transición de MTF
# Emplea todos los datos de entrenamiento (no por ventana)
transition_matrices = fit_all_transition_matrices(train_df_raw_encoding, scalers, feature_cols)

if 'MTF' in techniques:
  print('Matriz de transición MTF - Generación termoeléctrica con carbón')
  print(np.round(transition_matrices['generacion_carbon'], 3))

In [ ]:
# Construye imágenes de codificación para cada conjunto de datos
images_by_split = {}

for split_name in split_names:
  images_by_split[split_name] = encode_dataset(scaled_windows[split_name], transition_matrices,
                                               feature_cols, techniques, mtf_backend)

images_train = images_by_split['train']
images_val = images_by_split['val']
images_test = images_by_split['test']

example_var = feature_cols[0] # Variable de ejemplo

print('Formas resultantes')
for tech in techniques:
  print(f'{tech}: {images_train[example_var][tech].shape}')

if 'GASF' in techniques:
  gasf_all = images_train[example_var]['GASF']
  print('GASF simétrica en todas las ventanas:', np.allclose(gasf_all, gasf_all.transpose(0, 2, 1)))

if 'GADF' in techniques:
  gadf_all = images_train[example_var]['GADF']
  print('GADF antisimétrica en todas las ventanas:', np.allclose(gadf_all, -gadf_all.transpose(0, 2, 1)))

if 'RP' in techniques:
  rp_all = images_train[example_var]['RP']
  print('Diagonal cero de RP en todas las ventanas:', np.allclose(np.diagonal(rp_all, axis1 = 1, axis2 = 2), 0))

In [ ]:
# Selección de datos de prueba
idx_window = 400 # Ventana de ejemplo

# Rango de la ventana de tiempo
start_example_window = df_data_split_encoding['train']['window_start'][idx_window]
end_example_window = df_data_split_encoding['train']['window_end'][idx_window]
date_range = f"{pd.Timestamp(start_example_window).strftime('%Y-%m-%d')} - {pd.Timestamp(end_example_window).strftime('%Y-%m-%d')}"

In [ ]:
# Codificaciones visuales obtenidas para la variable y ventana de referencia
num_plots = len(techniques) + 1
num_cols = 3
num_rows = (num_plots + num_cols - 1) // num_cols # Calcula número de filas necesarias

fig, axs = plt.subplots(num_rows, num_cols, figsize = (12, 3.2 * num_rows))
axs = axs.flatten() # Aplana ejes para facilitar indexación

# Ventana de datos
raw_window = df_data_split_encoding['train']['X'][idx_window, :, feature_cols.index(example_var)]
axs[0].plot(raw_window, color = '#2563eb')
axs[0].set_title(f'{example_var}', fontsize = 10)
axs[0].grid(True) # Añade cuadrícula

# Grafica técnicas de codificación
for k, tech in enumerate(techniques):
  im = axs[k + 1].imshow(images_train[example_var][tech][idx_window], cmap = 'jet', aspect = 'equal')
  axs[k + 1].set_title(tech, fontsize = 10)
  axs[k + 1].set_xticks([]); axs[k + 1].set_yticks([])
  plt.colorbar(im, ax = axs[k + 1], fraction = 0.046, pad = 0.04)

# Oculta secciones no utilizadas
for i in range(num_plots, num_rows * num_cols):
  fig.delaxes(axs[i])

plt.suptitle(f'Técnicas de codificación visual\n Ventana: {idx_window} ({date_range})', fontweight = 'bold', y = 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Imagen codificada para todas las variables predictoras
num_plots = len(feature_cols)
num_cols = 3
num_rows = (num_plots + num_cols - 1) // num_cols # Calcula número de filas necesarias

fig, axs = plt.subplots(num_rows, num_cols, figsize = (12, 3.2 * num_rows))
axs = axs.flatten() # Aplana ejes para facilitar indexación

# Grafica representación visual de variables predictoras
for k, var in enumerate(feature_cols):
  im = axs[k].imshow(images_train[var][techniques[0]][idx_window], cmap = 'jet', aspect = 'equal')
  axs[k].set_title(var, fontsize = 10)
  axs[k].set_xticks([]); axs[k].set_yticks([])
  plt.colorbar(im, ax = axs[k], fraction = 0.046, pad = 0.04)

# Oculta secciones no utilizadas
for i in range(num_plots, num_rows * num_cols):
  fig.delaxes(axs[i])

plt.suptitle(f'Representación {techniques[0]} de variables predictoras\n Ventana: {idx_window} ({date_range})', fontweight = 'bold', y = 1.05)
plt.tight_layout()
plt.show()

## 7. Modelos convolucionales

### 7.1 Importación de librerías específicas

In [ ]:
# Importación de librerías específicas
from keras.applications import EfficientNetB0, MobileNetV2, ResNet50V2, VGG16
from keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from keras.applications.mobilenet_v2 import preprocess_input as mobilenetv2_preprocess
from keras.applications.resnet_v2 import preprocess_input as resnetv2_preprocess
from keras.applications.vgg16 import preprocess_input as vgg16_preprocess

### 7.2 Definición de variables

In [ ]:
# Modelos preentrenados a emplear
backbone_registry = {
    'EfficientNetB0': (EfficientNetB0, efficientnet_preprocess),
    'MobileNetV2': (MobileNetV2, mobilenetv2_preprocess),
    'ResNet50V2': (ResNet50V2, resnetv2_preprocess)}

# Tamaño objetivo de las imágenes
target_size = (128, 128)

### 7.3 Definición de funciones

In [ ]:
# Instancia backbone preentrenado (sin cabeza de clasificación)
# El renombrado posterior es necesario porque Keras exige nombres únicos al combinar
def get_backbone(backbone_name, input_shape, name = None):
  if backbone_name not in backbone_registry:
    raise ValueError(f'Backbone no soportado: {backbone_name}')

  ctor, _ = backbone_registry[backbone_name]
  base = ctor(include_top = False, weights = 'imagenet',
              input_shape = input_shape, pooling = None)

  if name is not None:
    base.name = name

  return base

# Retorna la función de preprocesamiento específica del backbone
def get_preprocess_function(backbone_name):
  _, preprocess_fn = backbone_registry[backbone_name]

  return preprocess_fn

# Remuestreo de imágenes a una resolución objetivo
def resize_images(images, target_size = target_size, method = 'bilinear'):
  images_tf = tf.convert_to_tensor(images, dtype = tf.float32)
  resized = tf.image.resize(images_tf, target_size, method = method)

  return resized.numpy()

# Contralo capas finales del backbone habilitadas para ajuste
def set_backbone_fine_tuning(backbone, fine_tune_layers = 0):
  if fine_tune_layers == 'all':
    backbone.trainable = True

    return backbone

  # Necesario para poder controlar capas individuales
  backbone.trainable = True
  n_layers = len(backbone.layers)
  freeze_until = max(0, n_layers - int(fine_tune_layers))

  for i, layer in enumerate(backbone.layers):
    layer.trainable = i >= freeze_until

    # BatchNormalization permanece congelado incluso dentro del rango entrenable
    # Con lotes pequeños, actualizar sus estadísticas desestabiliza lo aprendido en ImageNet
    if isinstance(layer, layers.BatchNormalization):
      layer.trainable = False

  return backbone

# Habilita el ajuste fino de las últimas capas de cada backbone del modelo híbrido y recompila
# Usar con learning_rate mucho menor que el de la fase inicial
def unfreeze_and_recompile(model, fine_tune_layers,
                           optimizer_name = 'adam',
                           learning_rate = 1e-5):
  backbones_found = 0

  for layer in model.layers:
    if isinstance(layer, tf.keras.Model): # Los backbones son sub-modelos anidados
      set_backbone_fine_tuning(layer, fine_tune_layers)
      backbones_found += 1

  if backbones_found == 0:
    raise ValueError('No se encuentran backbones en el modelo')

  model.compile(optimizer = get_optimizer(optimizer_name, learning_rate),
                loss = 'mae',
                metrics = [keras.metrics.RootMeanSquaredError(), 'mape'])

  return model

# Construye rama CNN por técnica de codificación
def build_technique_branch(input_shape, technique_name,
                           backbone_name = 'MobileNetV2',
                           target_size = target_size,
                           fine_tune_layers = 0,
                           pooling = 'avg'):
  n_channels = input_shape[-1]
  inputs = layers.Input(shape = input_shape, name = f'input_{technique_name}')

  # Adaptador de canales: N -> 3
  # Conv2D 1x1 infiere el número de canales de entrada
  x = layers.Conv2D(
      filters = 3, kernel_size = 1, padding = 'same', activation = None,
      name = f'{technique_name}_channel_adapter_{n_channels}_to_3')(inputs)

  # Remuestreo a la resolución que espera el backbone
  x = layers.Resizing(
      target_size[0], target_size[1], interpolation = 'bilinear',
      name = f'{technique_name}_resize')(x)

  # Preprocesamiento específico del backbone (normalización de intensidades)
  preprocess_fn = get_preprocess_function(backbone_name)
  x = layers.Lambda(preprocess_fn, name = f'{technique_name}_preprocess')(x)

  # Backbone preentrenado (instancia independiente por rama/técnica)
  # Nombre único para evitar colisiones al combinar varias ramas
  backbone = get_backbone(
      backbone_name, input_shape = (*target_size, 3),
      name = f'{technique_name}_{backbone_name.lower()}')

  set_backbone_fine_tuning(backbone, fine_tune_layers)
  x = backbone(x)

  if pooling == 'avg':
    x = layers.GlobalAveragePooling2D(name = f'{technique_name}_gap')(x)
  elif pooling == 'max':
    x = layers.GlobalMaxPooling2D(name = f'{technique_name}_gmp')(x)
  else:
    raise ValueError(f'Error. Tipo de pooling incorrecto: {pooling}')

  return inputs, x

# Fusión de ramas paralelas + cabeza de pronóstico
def fuse_branches(branch_outputs, fusion_method = 'concat'):
  # Combina los vectores de características de todas las ramas (técnicas) en uno solo
  if len(branch_outputs) == 1:
    return branch_outputs[0]

  if fusion_method == 'concat':
    return layers.Concatenate(name = 'fusion_concat')(branch_outputs)
  elif fusion_method == 'attention':
    # Cada rama recibe un peso aprendido (softmax entre ramas)
    # Permite que el modelo priorice, por ejemplo, la técnica más informativa
    stacked = layers.Lambda(
        lambda xs: tf.stack(xs, axis = 1), name = 'stack_branches')(branch_outputs)

    attention_scores = layers.Dense(1, name = 'attention_scores')(stacked)
    attention_weights = layers.Softmax(axis = 1, name = 'attention_weights')(attention_scores)

    fused = layers.Lambda(
        lambda x: tf.reduce_sum(x[0] * x[1], axis = 1),
        name = 'attention_fusion')([stacked, attention_weights])

    return fused
  else:
    raise ValueError(f'Error. Método de fusión incorrecto {fusion_method}')

# Construye modelo híbrido con una rama paralela por técnica de codificación
# Las ramas son fusionadas en un único vector seguido de una cabeza de pronóstico
def build_hybrid_model(input_shapes_by_technique, horizon, backbone_name = 'MobileNetV2',
                       target_size = target_size, fine_tune_layers = 0, pooling = 'avg',
                       fusion_method = 'concat', dense_units = (64, ), dropout_rate = 0.3,
                       optimizer_name = 'adam', learning_rate = 1e-4):
  branch_inputs = []
  branch_outputs = []

  # Rama independiente por técnica de codificación
  for technique_name, input_shape in input_shapes_by_technique.items():
    inp, out = build_technique_branch(
        input_shape, technique_name, backbone_name = backbone_name, target_size = target_size,
        fine_tune_layers = fine_tune_layers, pooling = pooling)

    branch_inputs.append(inp)
    branch_outputs.append(out)

  # Fusión de ramas
  fused = fuse_branches(branch_outputs, fusion_method = fusion_method)
  x = fused

  for j, units in enumerate(dense_units):
    x = layers.Dense(units, activation = 'relu', name = f'fusion_dense_{j + 1}')(x)
    x = layers.Dropout(dropout_rate, name = f'fusion_dropout_{j + 1}')(x)

  # Capas de salida
  outputs = layers.Dense(horizon, activation = 'linear', name = 'forecast_output')(x)

  # Construye el modelo
  model = tf.keras.Model(
      inputs = branch_inputs, outputs = outputs,
      name = f'Convolucional_{backbone_name.lower()}_{fusion_method}')

  # Compila el modelo
  model.compile(
      optimizer = get_optimizer(optimizer_name, learning_rate),
      loss = 'mae',
      metrics = [tf.keras.metrics.RootMeanSquaredError(), 'mape'])

  return model

### 7.4 Preparación de tensores de entrada

In [ ]:
# Apila las imágenes de codificación de las variables predictoras por técnica
def stack_variable_channels(images_by_variable, technique,
                            feature_cols = feature_cols):
  arrays = [images_by_variable[var][technique] for var in feature_cols]

  return np.stack(arrays, axis = -1)

# Construye diccionario de tensores de entrada para un conjunto de datos completo
def prepare_model_inputs(images_by_variable, techniques = techniques,
                         feature_cols = feature_cols):
  return {tech: stack_variable_channels(images_by_variable, tech, feature_cols) for tech in techniques}

# Obtiene {technique: (H, W, n_variables)} a partir de un diccionario de tensores ya preparado
def infer_input_shapes(model_inputs):
  return {tech: arr.shape[1:] for tech, arr in model_inputs.items()}

# Convierte {tecnica: array} al diccionario {'input_TECNICA': array} que espera Keras
# Usa diccionario para que Keras empareje cada arreglo con su capa Input por nombre
def assemble_hybrid_inputs(model_inputs, prefix = 'input_'):
  return {f'{prefix}{technique}': np.asarray(array, dtype = 'float32')
          for technique, array in model_inputs.items()}

In [ ]:
# Construye tensores de entrada por técnica para cada conjunto de datos
model_inputs_train = prepare_model_inputs(images_train, techniques, feature_cols)
model_inputs_val = prepare_model_inputs(images_val, techniques, feature_cols)
model_inputs_test = prepare_model_inputs(images_test, techniques, feature_cols)

# Formas de entrada por técnica: {tecnica: (alto, ancho, n_variables)}
input_shapes_by_technique = infer_input_shapes(model_inputs_train)

print('Formas de entrada por técnica:')
for tech, shape in input_shapes_by_technique.items():
  print(f'{tech}: {shape}')

### 7.5 Confección de modelos

In [ ]:
# Combina historial de las dos fases de entrenamiento en un solo objeto
class CombinedHistory:
  def __init__(self, history_phase1, history_phase2 = None):
    self.history = {}

    for key in ['loss', 'val_loss', 'mape', 'val_mape']:
      values = list(history_phase1.history.get(key, []))

      if history_phase2 is not None:
        values += list(history_phase2.history.get(key, []))

      self.history[key] = values

    self.epoch = list(range(len(self.history['loss'])))

# Entrena modelo híbrido en dos fases
# Fase 1 (extracción de características): Backbones congelados, tasa de aprendizaje normal
# Fase 2 (ajuste fino, opcional): últimas N capas habilitadas, tasa de aprendizaje mucho menor
def train_hybrid_model_two_phase(
    input_shapes_by_technique,
    train_inputs, train_target, val_inputs, val_target,
    model_name = 'Convolucional', backbone_name = 'MobileNetV2',
    fusion_method = 'concat', dense_units = (64, ), dropout_rate = 0.3,
    phase1_optimizer = 'adam', phase1_learning_rate = 1e-4,
    phase1_epochs = EPOCHS, run_phase2 = True,
    phase2_fine_tune_layers = 15, phase2_optimizer = 'adam',
    phase2_learning_rate = 1e-5, phase2_epochs = EPOCHS,
    batch_size = BATCH_SIZE):
  # Confección del modelo
  model = build_hybrid_model(
      input_shapes_by_technique, horizon = horizon,
      backbone_name = backbone_name, target_size = target_size,
      fine_tune_layers = 0, pooling = 'avg',
      fusion_method = fusion_method, dense_units = dense_units,
      dropout_rate = dropout_rate, optimizer_name = phase1_optimizer,
      learning_rate = phase1_learning_rate)

  print(f'Fase 1: Extracción de características ({model_name})')
  history_phase1 = model.fit(
      train_inputs, train_target,
      validation_data = (val_inputs, val_target),
      epochs = phase1_epochs,
      batch_size = batch_size,
      callbacks = build_callbacks(f'{model_name}_fase1'),
      verbose = 1)

  history_phase2 = None

  if run_phase2:
    unfreeze_and_recompile(
        model, fine_tune_layers = phase2_fine_tune_layers,
        optimizer_name = phase2_optimizer,
        learning_rate = phase2_learning_rate)

    print(f'Fase 2: Ajuste fino de últimas {phase2_fine_tune_layers} capas')
    history_phase2 = model.fit(
        train_inputs, train_target,
        validation_data = (val_inputs, val_target),
        epochs = phase2_epochs,
        batch_size = batch_size,
        callbacks = build_callbacks(f'{model_name}_fase2'),
        verbose = 1)

  return model, history_phase1, history_phase2

# Ejecuta el flujo completo para configuración de modelo híbrido
def run_hybrid_experiment(
    input_shapes_by_technique, model_inputs_train, y_train,
    model_inputs_val, y_val,
    model_inputs_test, y_test, anchors_test,
    model_name = 'Convolucional', backbone_name = 'MobileNetV2',
    fusion_method = 'concat', dense_units = (64, ), dropout_rate = 0.3,
    phase1_optimizer = 'adam', phase1_learning_rate = 1e-4,
    phase1_epochs = EPOCHS, run_phase2 = True,
    phase2_fine_tune_layers = 15, phase2_optimizer = 'adam',
    phase2_learning_rate = 1e-5, phase2_epochs = EPOCHS,
    batch_size = BATCH_SIZE):
  keras.utils.set_random_seed(SEED) # Garantiza reproducibilidad

  train_inputs = assemble_hybrid_inputs(model_inputs_train)
  val_inputs = assemble_hybrid_inputs(model_inputs_val)
  test_inputs = assemble_hybrid_inputs(model_inputs_test)

  # Ajuste (entrenamiento) del modelo
  model, history_phase1, history_phase2 = train_hybrid_model_two_phase(
      input_shapes_by_technique, train_inputs, y_train, val_inputs, y_val,
      model_name = model_name, backbone_name = backbone_name, fusion_method = fusion_method,
      dense_units = dense_units, dropout_rate = dropout_rate,
      phase1_optimizer = phase1_optimizer, phase1_learning_rate = phase1_learning_rate,
      phase1_epochs = phase1_epochs, run_phase2 = run_phase2,
      phase2_fine_tune_layers = phase2_fine_tune_layers, phase2_optimizer = phase2_optimizer,
      phase2_learning_rate = phase2_learning_rate, phase2_epochs = phase2_epochs,
      batch_size = batch_size)

  # Historial de entrenamiento (ambas fases)
  print('\n')
  training_history_plot(CombinedHistory(history_phase1, history_phase2))

  # Predicción sobre conjunto de prueba (en escala log-diff)
  pred_test = model.predict(test_inputs, verbose = 0)

  # Revierte transformación log-diff de la variable objetivo
  y_test_original = to_original_scale(y_test, anchors_test)
  pred_test_original = to_original_scale(pred_test, anchors_test)

  # Métricas de evaluación
  print('\n')
  results = dict(forecast_metrics(model_name, y_test_original, pred_test_original))

  # Predicción vs real (en escala original)
  run_conformal_bands(
      model_name,
      y_test_original,
      pred_test_original)

  # Análisis de residuos (real - predicho)
  print('\n')
  residuals_analysis(
      model_name,
      y_test_original,
      pred_test_original,
      horizon_dates = get_horizon_dates('test'))

  return model

### 7.6 Entrenamiento y validación

In [ ]:
chosen_model = 'EfficientNetB0'
fusion_method = 'concat'

In [ ]:
# Ejecuta el flujo completo para configuración de modelo híbrido

if 'model_hybrid' in globals():
  del model_hybrid

keras.backend.clear_session() # Libera el grafo y reinicia los contadores de nombres de capas
gc.collect()

model_hybrid = run_hybrid_experiment(
    input_shapes_by_technique,
    model_inputs_train, y_train,
    model_inputs_val, y_val,
    model_inputs_test, y_test, anchors_test,
    model_name = f'Convolucional_{chosen_model}_{fusion_method}',
    backbone_name = chosen_model,
    fusion_method = fusion_method,
    run_phase2 = True,
    phase2_fine_tune_layers = 15)

## 8. Modelos ViT

### 8.1 Definición de variables

In [ ]:
vit_image_size = 60 # Resolución a la que se remuestrean todas las técnicas (debe ser divisible por patch_size)
patch_size = 10 # Tamaño de parche: genera (60/10)^2 = 36 parches por imagen
projection_dim = 64 # Dimensión del espacio de proyección de cada parche
num_heads = 4 # Número de cabezas de atención
transformer_layers = 4 # Número de bloques Transformer apilados
mlp_ratio = 2 # Factor de expansión del perceptrón interno (projection_dim * mlp_ratio)
vit_dropout_rate = 0.1 # Tasa de dropout dentro de los bloques Transformer

# Se mantiene deliberadamente pequeño: los Vision Transformer carecen del sesgo inductivo de localidad de las CNN
print(f'Parches por imagen: {(vit_image_size // patch_size) ** 2}')

### 8.2 Definición de funciones

In [ ]:
# Extrae parches y los proyecta a un espacio de dimensión fija
# Proyección con Conv2D con stride igual al tamaño de parche
# Acepta cualquier número de canales de entrada
class PatchEmbedding(layers.Layer):
  def __init__(self, patch_size, projection_dim, **kwargs):
    super().__init__(**kwargs)
    self.patch_size = patch_size
    self.projection_dim = projection_dim
    self.projection = layers.Conv2D(
        filters = projection_dim, kernel_size = patch_size, strides = patch_size,
        padding = 'valid', name = f'{self.name}_projection')

  def call(self, inputs):
    patches = self.projection(inputs) # (batch, alto/parche, ancho/parche, projection_dim)
    batch_size = ops.shape(patches)[0]
    n_patches = ops.shape(patches)[1] * ops.shape(patches)[2]

    return ops.reshape(patches, (batch_size, n_patches, self.projection_dim))

  def get_config(self):
    config = super().get_config()
    config.update({'patch_size': self.patch_size, 'projection_dim': self.projection_dim})

    return config

# Agrega codificación posicional aprendida a la secuencia de parches
# El Transformer no tiene noción intrínseca de posición (sin esto, la ubicación de cada parche dentro de la imagen se pierde)
class PositionEmbedding(layers.Layer):
  def __init__(self, n_patches, projection_dim, **kwargs):
    super().__init__(**kwargs)
    self.n_patches = n_patches
    self.projection_dim = projection_dim
    self.position_embedding = layers.Embedding(
        input_dim = n_patches, output_dim = projection_dim,
        name = f'{self.name}_position')

  def call(self, inputs):
    positions = ops.arange(0, self.n_patches, dtype = 'int32')

    return inputs + self.position_embedding(positions)

  def get_config(self):
    config = super().get_config()
    config.update({'n_patches': self.n_patches, 'projection_dim': self.projection_dim})

    return config

# Bloque Transformer: atención multi-cabeza más perceptrón (ambos con conexión residual)
# Emplea normalización previa (más estable durante entrenamiento)
def transformer_block(x, num_heads, projection_dim, mlp_units, dropout_rate, block_name):
  # Atención multi-cabeza
  x_norm = layers.LayerNormalization(epsilon = 1e-6, name = f'{block_name}_ln1')(x)
  attention = layers.MultiHeadAttention(
      num_heads = num_heads, key_dim = projection_dim // num_heads,
      dropout = dropout_rate, name = f'{block_name}_mha')(x_norm, x_norm)
  x = layers.Add(name = f'{block_name}_add1')([x, attention])

  # Perceptrón multicapa
  x_norm = layers.LayerNormalization(epsilon = 1e-6, name = f'{block_name}_ln2')(x)
  mlp = layers.Dense(mlp_units, activation = 'gelu', name = f'{block_name}_mlp1')(x_norm)
  mlp = layers.Dropout(dropout_rate, name = f'{block_name}_mlp_do1')(mlp)
  mlp = layers.Dense(projection_dim, name = f'{block_name}_mlp2')(mlp)
  mlp = layers.Dropout(dropout_rate, name = f'{block_name}_mlp_do2')(mlp)

  return layers.Add(name = f'{block_name}_add2')([x, mlp])

# Construye rama Vision Transformer para una técnica de codificación
# Las imágenes se remuestrean a vit_image_size para que el tamaño de parche divida exactamente la imagen
# Homogenizando imágenes de técnicas con diferente resolución
def build_vit_branch(input_shape, technique_name, vit_image_size = vit_image_size,
                     patch_size = patch_size, projection_dim = projection_dim,
                     num_heads = num_heads, transformer_layers = transformer_layers,
                     mlp_ratio = mlp_ratio, dropout_rate = vit_dropout_rate):
  if vit_image_size % patch_size != 0:
    raise ValueError(f'vit_image_size ({vit_image_size}) debe ser divisible por patch_size ({patch_size})')

  n_patches = (vit_image_size // patch_size) ** 2
  inputs = layers.Input(shape = input_shape, name = f'input_{technique_name}')

  # Homogeniza resolución de imágenes de las técnicas de codificación
  x = layers.Resizing(
      vit_image_size, vit_image_size, interpolation = 'bilinear',
      name = f'{technique_name}_resize')(inputs)

  # Secuencia de parches proyectados más codificación posicional
  x = PatchEmbedding(patch_size, projection_dim, name = f'{technique_name}_patch_embed')(x)
  x = PositionEmbedding(n_patches, projection_dim, name = f'{technique_name}_pos_embed')(x)
  x = layers.Dropout(dropout_rate, name = f'{technique_name}_embed_do')(x)

  # Pila de bloques Transformer
  for i in range(transformer_layers):
    x = transformer_block(
        x, num_heads, projection_dim, projection_dim * mlp_ratio, dropout_rate,
        block_name = f'{technique_name}_tb{i + 1}')

  x = layers.LayerNormalization(epsilon = 1e-6, name = f'{technique_name}_ln_final')(x)

  # Agregación de parches en un único vector de características por técnica
  x = layers.GlobalAveragePooling1D(name = f'{technique_name}_gap')(x)

  return inputs, x

# Construye modelo ViT con una rama paralela por técnica de codificación
# Las ramas son fusionadas en un único vector, seguido de la cabeza de pronóstico
def build_vit_model(input_shapes_by_technique, horizon, vit_image_size = vit_image_size,
                    patch_size = patch_size, projection_dim = projection_dim,
                    num_heads = num_heads, transformer_layers = transformer_layers,
                    mlp_ratio = mlp_ratio, dropout_rate = vit_dropout_rate,
                    fusion_method = 'concat', dense_units = (64, ),
                    head_dropout_rate = 0.3, optimizer_name = 'adam', learning_rate = 1e-4):
  branch_inputs = []
  branch_outputs = []

  # Rama independiente por técnica de codificación
  for technique_name, input_shape in input_shapes_by_technique.items():
    inp, out = build_vit_branch(
        input_shape, technique_name, vit_image_size = vit_image_size, patch_size = patch_size,
        projection_dim = projection_dim, num_heads = num_heads,
        transformer_layers = transformer_layers, mlp_ratio = mlp_ratio,
        dropout_rate = dropout_rate)

    branch_inputs.append(inp)
    branch_outputs.append(out)

  # Fusión de ramas
  fused = fuse_branches(branch_outputs, fusion_method = fusion_method)
  x = fused

  for j, units in enumerate(dense_units):
    x = layers.Dense(units, activation = 'relu', name = f'fusion_dense_{j + 1}')(x)
    x = layers.Dropout(head_dropout_rate, name = f'fusion_dropout_{j + 1}')(x)

  # Capas de salida
  outputs = layers.Dense(horizon, activation = 'linear', name = 'forecast_output')(x)

  # Construye el modelo
  model = models.Model(
      inputs = branch_inputs,
      outputs = outputs,
      name = f'vit_{fusion_method}')

  # Compila el modelo
  model.compile(
      optimizer = get_optimizer(optimizer_name, learning_rate),
      loss = 'mae',
      metrics = [keras.metrics.RootMeanSquaredError(), 'mape'])

  return model

### 8.3 Confección de modelos

In [ ]:
# Entrena modelo ViT y lo evalúa sobre el conjunto de prueba
def run_vit_experiment(
    input_shapes_by_technique,
    model_inputs_train, y_train,
    model_inputs_val, y_val,
    model_inputs_test, y_test,
    anchors_test, model_name = 'ViT',
    fusion_method = 'concat', dense_units = (64, ),
    head_dropout_rate = 0.3, optimizer_name = 'adam',
    learning_rate = 1e-4, epochs = EPOCHS,
    batch_size = BATCH_SIZE):
  keras.utils.set_random_seed(SEED) # Garantiza reproducibilidad

  train_inputs = assemble_hybrid_inputs(model_inputs_train)
  val_inputs = assemble_hybrid_inputs(model_inputs_val)
  test_inputs = assemble_hybrid_inputs(model_inputs_test)

  # Confección de modelo
  model = build_vit_model(
      input_shapes_by_technique, horizon = horizon,
      fusion_method = fusion_method, dense_units = dense_units,
      head_dropout_rate = head_dropout_rate, optimizer_name = optimizer_name,
      learning_rate = learning_rate)

  # Ajuste (entrenamiento) del modelo
  print(f'Entrenamiento ({model_name})')
  history = model.fit(
      train_inputs,
      y_train,
      validation_data = (val_inputs, y_val),
      epochs = epochs,
      batch_size = batch_size,
      callbacks = build_callbacks(model_name),
      verbose = 1)

  # Historial de entrenamiento
  print('\n')
  training_history_plot(history)

  # Predicción sobre conjunto de prueba (en escala log-diff)
  pred_test = model.predict(test_inputs, verbose = 0)

  # Revierte transformación log-diff de la variable objetivo
  y_test_original = to_original_scale(y_test, anchors_test)
  pred_test_original = to_original_scale(pred_test, anchors_test)

  # Métricas de evaluación
  results = dict(forecast_metrics(model_name, y_test_original, pred_test_original))

  # Predicción vs real (en escala original)
  run_conformal_bands(
      model_name,
      y_test_original,
      pred_test_original)

  # Análisis de residuos (real - predicho)
  residuals_analysis(
      model_name,
      y_test_original,
      pred_test_original,
      horizon_dates = get_horizon_dates('test'))

  return model

### 8.4 Entrenamiento y validación

In [ ]:
fusion_method = 'concat'

In [ ]:
# Ejecuta el flujo completo para configuración de modelo ViT

if 'model_vit' in globals():
  del model_vit

keras.backend.clear_session() # Libera el grafo y reinicia los contadores de nombres de capas
gc.collect()

model_vit = run_vit_experiment(
    input_shapes_by_technique,
    model_inputs_train, y_train,
    model_inputs_val, y_val,
    model_inputs_test, y_test,
    anchors_test, model_name = f'ViT_{fusion_method}',
    fusion_method = fusion_method)

## 9. Modelos multimodales

Combinan dos modalidades de representación de la misma ventana temporal:

- **Rama secuencial**: Procesa la serie de tiempo mediante una
arquitectura recurrente. Admite variables adicionales de ingeniería de características como días festivos o transformaciones seno/coseno del calendario.

- **Ramas de imágenes**: Una rama paralela por técnica de codificación visual, empleando
arquitecturas convolucionales con transferencia de aprendizaje o Visión Transformer.

Todas las ramas se proyectan a una dimensión común, se fusionan en un único vector y alimentan
la cabeza de pronóstico. La proyección previa es necesaria porque la fusión por atención apila
las ramas y exige que todas tengan igual dimensión: la rama secuencial y las de imagen producen
vectores de tamaños muy distintos.

### 9.1 Definición de variables

In [ ]:
# Tipo de rama de imagen: 'cnn' (backbone preentrenado) o 'vit'
image_branch_type = 'cnn'

# Dimensión común a la que se proyecta cada rama antes de fusión
# Obligatoria para fusión por atención, que apila las ramas y exige igual dimensión
fusion_dim = 64

### 9.2 Definición de funciones

In [ ]:
# Proyecta una rama a la dimensión común de fusión
# La fusión por atención apila las ramas y exige que todas tengan igual dimensión
def project_branch(branch_output, branch_name, fusion_dim = fusion_dim):
  return layers.Dense(fusion_dim, activation = 'relu', name = f'{branch_name}_projection')(branch_output)

# Construye modelo multimodal
# Rama secuencial (serie temporal cruda) + una rama por técnica de codificación visual
def build_multimodal_model(seq_input_shape, input_shapes_by_technique,
                           horizon, use_sequential = True,
                           image_branch_type = image_branch_type, backbone_name = 'MobileNetV2',
                           target_size = target_size, fine_tune_layers = 0,
                           pooling = 'avg', branch_type = seq_branch_type,
                           seq_units = seq_units, seq_layers = seq_layers,
                           seq_dropout_rate = seq_dropout_rate, fusion_method = 'concat',
                           fusion_dim = fusion_dim, dense_units = (64, ),
                           dropout_rate = 0.3, optimizer_name = 'adam',
                           learning_rate = 1e-4):
  branch_inputs = []
  branch_outputs = []

  # Rama secuencial (opcional: permite comparar contra el modelo solo de imágenes)
  if use_sequential:
    seq_inp, seq_out = build_sequential_branch(
        seq_input_shape, branch_type = branch_type, units = seq_units,
        n_layers = seq_layers, dropout_rate = seq_dropout_rate)

    branch_inputs.append(seq_inp)
    branch_outputs.append(project_branch(seq_out, 'sequential', fusion_dim))

  # Ramas de imágenes (una por técnica de codificación visual)
  for technique_name, input_shape in input_shapes_by_technique.items():
    if image_branch_type == 'cnn':
      img_inp, img_out = build_technique_branch(
          input_shape, technique_name, backbone_name = backbone_name,
          target_size = target_size, fine_tune_layers = fine_tune_layers, pooling = pooling)
    elif image_branch_type == 'vit':
      img_inp, img_out = build_vit_branch(input_shape, technique_name)
    else:
      raise ValueError(f'Error. Tipo de rama de imagen incorrecto {image_branch_type}')

    branch_inputs.append(img_inp)
    branch_outputs.append(project_branch(img_out, technique_name, fusion_dim))

  if len(branch_inputs) == 0:
    raise ValueError('Error. El modelo requiere al menos una rama')

  fused = fuse_branches(branch_outputs, fusion_method = fusion_method)
  x = fused

  for j, units in enumerate(dense_units):
    x = layers.Dense(units, activation = 'relu', name = f'head_dense_{j + 1}')(x)
    x = layers.Dropout(dropout_rate, name = f'head_dropout_{j + 1}')(x)

  # Capas de salida
  outputs = layers.Dense(horizon, activation = 'linear', name = 'forecast_output')(x)

  # Construye el modelo
  model = models.Model(
      inputs = branch_inputs,
      outputs = outputs,
      name = f'multimodal_{image_branch_type}_{fusion_method}')

  # Compila el modelo
  model.compile(
      optimizer = get_optimizer(optimizer_name, learning_rate),
      loss = 'mae',
      metrics = [keras.metrics.RootMeanSquaredError(), 'mape'])

  return model

# Ensambla las entradas del modelo multimodal
# Keras empareja cada arreglo con su capa Input por nombre, sin depender del orden
def assemble_multimodal_inputs(X_seq, model_inputs, use_sequential = True, prefix = 'input_'):
  inputs = {f'{prefix}{technique}': np.asarray(array, dtype = 'float32')
            for technique, array in model_inputs.items()}

  if use_sequential:
    inputs['input_sequential'] = np.asarray(X_seq, dtype = 'float32')

  return inputs

### 9.3 Confección de modelos

In [ ]:
# Entrena el modelo multimodal en dos fases
# Fase 1 (extracción de características): backbones congelados, tasa de aprendizaje inicial
# Fase 2 (ajuste fino, opcional): últimas N capas habilitadas, tasa de aprendizaje menor
# Con ramas 'ViT' se entrena en una fase (no hay pesos preentrenados que preservar)
def train_multimodal_two_phase(
    seq_input_shape, input_shapes_by_technique,
    train_inputs, train_target, val_inputs, val_target,
    model_name = 'Multimodal', use_sequential = True,
    image_branch_type = image_branch_type, backbone_name = 'MobileNetV2',
    branch_type = seq_branch_type, seq_units = seq_units,
    seq_layers = seq_layers, seq_dropout_rate = seq_dropout_rate,
    fusion_method = 'concat', fusion_dim = fusion_dim,
    dense_units = (64, ), dropout_rate = 0.3,
    phase1_optimizer = 'adam', phase1_learning_rate = 1e-4,
    phase1_epochs = EPOCHS, run_phase2 = True,
    phase2_fine_tune_layers = 15, phase2_optimizer = 'adam',
    phase2_learning_rate = 1e-5, phase2_epochs = EPOCHS,
    batch_size = BATCH_SIZE):
  # Confección del modelo
  model = build_multimodal_model(
      seq_input_shape, input_shapes_by_technique,
      horizon = horizon, use_sequential = use_sequential,
      image_branch_type = image_branch_type, backbone_name = backbone_name,
      target_size = target_size, fine_tune_layers = 0,
      pooling = 'avg', branch_type = branch_type,
      seq_units = seq_units, seq_layers = seq_layers,
      seq_dropout_rate = seq_dropout_rate, fusion_method = fusion_method,
      fusion_dim = fusion_dim, dense_units = dense_units,
      dropout_rate = dropout_rate, optimizer_name = phase1_optimizer,
      learning_rate = phase1_learning_rate)

  print(f'Fase 1: Extracción de características ({model_name})')
  history_phase1 = model.fit(
      train_inputs, train_target,
      validation_data = (val_inputs, val_target),
      epochs = phase1_epochs,
      batch_size = batch_size,
      callbacks = build_callbacks(f'{model_name}_fase1'),
      verbose = 1)

  history_phase2 = None

  # Ajuste fino aplicado a backbones preentrenados
  if run_phase2 and image_branch_type == 'cnn':
    unfreeze_and_recompile(
        model, fine_tune_layers = phase2_fine_tune_layers,
        optimizer_name = phase2_optimizer, learning_rate = phase2_learning_rate)

    print(f'Fase 2: Ajuste fino de últimas {phase2_fine_tune_layers} capas')
    history_phase2 = model.fit(
        train_inputs, train_target,
        validation_data = (val_inputs, val_target),
        epochs = phase2_epochs,
        batch_size = batch_size,
        callbacks = build_callbacks(f'{model_name}_fase2'),
        verbose = 1)

  return model, history_phase1, history_phase2

# Ejecuta el flujo completo para una configuración de modelo multimodal
def run_multimodal_experiment(
    seq_input_shape, input_shapes_by_technique,
    X_seq_train, model_inputs_train, y_train,
    X_seq_val, model_inputs_val, y_val,
    X_seq_test, model_inputs_test, y_test,
    anchors_test, model_name = 'Multimodal',
    use_sequential = True, image_branch_type = image_branch_type,
    backbone_name = 'MobileNetV2', branch_type = seq_branch_type,
    seq_units = seq_units, seq_layers = seq_layers,
    seq_dropout_rate = seq_dropout_rate, fusion_method = 'concat',
    fusion_dim = fusion_dim, dense_units = (64, ),
    dropout_rate = 0.3, phase1_optimizer = 'adam',
    phase1_learning_rate = 1e-4, phase1_epochs = EPOCHS,
    run_phase2 = True, phase2_fine_tune_layers = 15,
    phase2_optimizer = 'adam', phase2_learning_rate = 1e-5,
    phase2_epochs = EPOCHS, batch_size = BATCH_SIZE):
  keras.utils.set_random_seed(SEED) # Garantiza reproducibilidad

  train_inputs = assemble_multimodal_inputs(X_seq_train, model_inputs_train, use_sequential)
  val_inputs = assemble_multimodal_inputs(X_seq_val, model_inputs_val, use_sequential)
  test_inputs = assemble_multimodal_inputs(X_seq_test, model_inputs_test, use_sequential)

  # Ajuste (entrenamiento) del modelo
  model, history_phase1, history_phase2 = train_multimodal_two_phase(
      seq_input_shape, input_shapes_by_technique,
      train_inputs, y_train, val_inputs, y_val,
      model_name = model_name, use_sequential = use_sequential,
      image_branch_type = image_branch_type, backbone_name = backbone_name,
      branch_type = branch_type, seq_units = seq_units,
      seq_layers = seq_layers, seq_dropout_rate = seq_dropout_rate,
      fusion_method = fusion_method, fusion_dim = fusion_dim,
      dense_units = dense_units, dropout_rate = dropout_rate,
      phase1_optimizer = phase1_optimizer, phase1_learning_rate = phase1_learning_rate,
      phase1_epochs = phase1_epochs, run_phase2 = run_phase2,
      phase2_fine_tune_layers = phase2_fine_tune_layers, phase2_optimizer = phase2_optimizer,
      phase2_learning_rate = phase2_learning_rate, phase2_epochs = phase2_epochs,
      batch_size = batch_size)

  # Historial de entrenamiento (ambas fases)
  print('\n')
  training_history_plot(CombinedHistory(history_phase1, history_phase2))

  # Predicción sobre conjunto de prueba (en escala log-diff)
  pred_test = model.predict(test_inputs, verbose = 0)

  # Revierte transformación log-diff de la variable objetivo
  y_test_original = to_original_scale(y_test, anchors_test)
  pred_test_original = to_original_scale(pred_test, anchors_test)

  # Métricas de evaluación
  print('\n')
  results = dict(forecast_metrics(model_name, y_test_original, pred_test_original))

  # Predicción vs real (en escala original)
  run_conformal_bands(
      model_name,
      y_test_original,
      pred_test_original)

  # Análisis de residuos (real - predicho)
  print('\n')
  residuals_analysis(
      model_name,
      y_test_original,
      pred_test_original,
      horizon_dates = get_horizon_dates('test'))

  return model

### 9.4 Entrenamiento y validación

In [ ]:
# Configuración del experimento
chosen_model = 'EfficientNetB0' # Backbone convolucional (solo si image_branch_type es 'cnn')
fusion_method = 'concat' # concat o attention
seq_branch_type = 'CNN-LSTM' # LSTM, GRU, BiLSTM, CNN-LSTM

In [ ]:
# Ejecuta el flujo completo para configuración de modelo multimodal

if 'model_multimodal' in globals():
  del model_multimodal

keras.backend.clear_session() # Libera el grafo y reinicia los contadores de nombres de capas
gc.collect()

# Ejecuta el flujo completo para configuración de modelo multimodal
model_multimodal = run_multimodal_experiment(
    seq_input_shape, input_shapes_by_technique,
    X_seq_train, model_inputs_train, y_train,
    X_seq_val, model_inputs_val, y_val,
    X_seq_test, model_inputs_test, y_test,
    anchors_test, model_name = f'Multimodal_{seq_branch_type}_{chosen_model}_{fusion_method}',
    use_sequential = True, image_branch_type = image_branch_type, # Definida en variables (CNN o ViT)
    backbone_name = chosen_model, branch_type = seq_branch_type, # Definida en variables (LSTM, GRU, BiLSTM, CNN-LSTM)
    fusion_method = fusion_method, run_phase2 = True,
    phase2_fine_tune_layers = 15)
